# HW3a: Conditional GAN for synthetic PneumoniaMNIST + Ray Tune + Ray Serve

> **Safety boundary.** Every generated image here is **SYNTHETIC**. It is for education and research prototyping only.
> It is not real patient data and not clinical evidence.

**How to run (Cursor + Colab extension, A100):** select the Colab kernel with an A100 runtime, then **Run All**.
The notebook writes the whole project into `/content/hw3a` (the `%%writefile` cells in section 0) and runs every step.
Approximate run time: 45–75 min on an A100/L4, 1.5–2 h on a T4 (sizes are chosen automatically in section 0).
All figures and tables are displayed inline, so the saved notebook keeps the results. The last cell builds the report
and zips every output.

| Step | Section |
|---|---|
| Setup, source, unit tests | 0 |
| 1. Data inspection | 1 |
| 2. Baseline cDCGAN, plus a deliberately unstable run | 2 |
| 3. Evaluation protocol (judge classifier) | 3 |
| 4–5. Ray Tune smoke test + full ASHA search | 4 |
| 6. Select, stress-test, downstream comparison | 5 |
| 7–8. Ray Serve deploy, client, load test, autoscaling | 6 |
| Report, model card, reproducibility, packaging | 7 |

## 0. Setup

In [ ]:
!nvidia-smi
import os, sys
PROJECT = "/content/hw3a" if os.path.exists("/content") else os.path.abspath("hw3a")
os.makedirs(PROJECT, exist_ok=True)
for d in ["gan_lab", "tests", "report"]:
    os.makedirs(os.path.join(PROJECT, d), exist_ok=True)
%cd {PROJECT}
print(sys.version)

In [ ]:
# Colab already has a CUDA build of torch/torchvision. Install the rest of the stack.
!pip install -q "ray[tune,serve]" medmnist tensorboardX markdown
import torch, ray
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("ray", ray.__version__, "| CPUs:", os.cpu_count())

In [ ]:
# Run settings chosen from the hardware. Big GPUs (A100/L4/H100) get the full study.
# Smaller ones (T4, V100) get a shorter but still complete search. Edit here to override.
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
NCPU = os.cpu_count() or 2
BIG = any(k in GPU for k in ("A100", "H100", "L4", "RTX", "5090", "4090", "3090"))
TUNE_SAMPLES = 24 if BIG else 16
TUNE_EPOCHS  = 40 if BIG else 30
BASE_EPOCHS  = 60 if BIG else 40
UNSTABLE_EPOCHS = 40 if BIG else 25
# Ray CPU accounting: Colab T4 machines have only 2 vCPUs, so request fractional CPUs.
# That way 4 tuning trials can share the GPU and Serve can still scale to 4 replicas.
CPT = max(1, NCPU // 4) if NCPU >= 8 else 0.5
SERVE_CPUS = 1 if NCPU >= 8 else 0.4
print(dict(GPU=GPU, CPUs=NCPU, TUNE_SAMPLES=TUNE_SAMPLES, TUNE_EPOCHS=TUNE_EPOCHS, BASE_EPOCHS=BASE_EPOCHS,
           CPT=CPT, SERVE_CPUS=SERVE_CPUS))

### 0.1 Project source
These cells write the project files to the runtime. They are collapsed by default; expand them to read the code.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=61"]
build-backend = "setuptools.build_meta"

[project]
name = "gan_lab"
version = "1.0.0"
description = "Conditional GAN for synthetic PneumoniaMNIST + Ray Tune + Ray Serve (education only)"
requires-python = ">=3.10"

[tool.setuptools]
packages = ["gan_lab"]

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["."]

In [ ]:
%%writefile requirements.txt
# Install PyTorch first for your platform: https://pytorch.org/get-started/locally/
torch>=2.1
torchvision>=0.16
ray[tune,serve]>=2.9
medmnist>=3.0
numpy
scipy
pandas
scikit-learn
matplotlib
pillow
fastapi
uvicorn
pydantic>=2
requests
tensorboard
tensorboardX
markdown
pytest

In [ ]:
%%writefile gan_lab/__init__.py
"""Conditional GAN lab for synthetic PneumoniaMNIST images (education/research only).

Label mapping (from MedMNIST INFO): 0 = normal, 1 = pneumonia.
Generated images are SYNTHETIC and must never be presented as real patient data.
"""

__version__ = "1.0.0"

LABEL_NAMES = {0: "normal", 1: "pneumonia"}
NAME_TO_LABEL = {v: k for k, v in LABEL_NAMES.items()}

SYNTHETIC_DISCLAIMER = (
    "SYNTHETIC IMAGE generated by a conditional GAN trained on PneumoniaMNIST. "
    "For education and research prototyping only. Not real patient data, "
    "not clinical evidence, and not suitable for diagnosis or treatment decisions."
)

In [ ]:
%%writefile gan_lab/utils.py
"""Small shared helpers: seeding, devices, JSON IO, image grids."""
from __future__ import annotations

import hashlib
import json
import os
import random
from typing import Any

import numpy as np
import torch


def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed python, numpy and torch (CPU + CUDA)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device(prefer: str | None = None) -> torch.device:
    if prefer:
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _json_default(o: Any):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, torch.Tensor):
        return o.detach().cpu().tolist()
    return str(o)


def save_json(obj: Any, path: str) -> None:
    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=_json_default)


def load_json(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path


def to_uint8(x: torch.Tensor) -> np.ndarray:
    """[-1, 1] tensor (N,1,H,W) -> uint8 numpy (N,H,W)."""
    x = ((x.detach().float().cpu().clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)
    return x[:, 0].numpy()


def save_grid(x: torch.Tensor, path: str, nrow: int = 8, value_range=(-1, 1)) -> None:
    """Save a tensor batch as an image grid (expects GAN space [-1,1] by default)."""
    from torchvision.utils import save_image

    d = os.path.dirname(path)
    if d:
        os.makedirs(d, exist_ok=True)
    lo, hi = value_range
    save_image((x.detach().float().cpu() - lo) / (hi - lo), path, nrow=nrow, padding=2, pad_value=1.0)


def array_sha256(a: np.ndarray) -> str:
    return hashlib.sha256(np.ascontiguousarray(a).tobytes()).hexdigest()


def file_sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
%%writefile gan_lab/data.py
"""PneumoniaMNIST loading and preprocessing.

Everything is held as in-memory tensors (the dataset is tiny: 4708/524/624 images),
which makes GAN training fast and lets Ray Tune ship the data to trials through the
object store instead of every trial re-reading files.

Spaces used in this project
---------------------------
* ``x28``   : original 28x28 images scaled to [0, 1].
* GAN space : images resized to ``img_size`` (default 64, as in the assignment) and
              normalised to [-1, 1] so they match the generator's Tanh output.
* ``rt28``  : "round-trip" images (28 -> img_size -> 28). Synthetic images are produced
              at img_size and downsampled to 28 for the classifier. Real images used by
              classifiers / memorisation checks go through the same round-trip so that a
              downstream classifier cannot use "slightly blurrier = synthetic = normal"
              as a shortcut.
"""
from __future__ import annotations

import os
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn.functional as F

SPLITS = ("train", "val", "test")


@dataclass
class SplitData:
    x28: torch.Tensor  # float32 (N,1,28,28) in [0,1]
    y: torch.Tensor  # int64 (N,)  0=normal, 1=pneumonia

    def __len__(self) -> int:
        return int(self.y.shape[0])


def get_label_mapping() -> dict[int, str]:
    """Label mapping straight from the MedMNIST metadata (do not hard-code blindly)."""
    from medmnist import INFO

    return {int(k): v for k, v in INFO["pneumoniamnist"]["label"].items()}


def load_pneumoniamnist(root: str = "data", download: bool = True) -> dict[str, SplitData]:
    from medmnist import PneumoniaMNIST

    root = os.path.abspath(root)
    os.makedirs(root, exist_ok=True)
    out: dict[str, SplitData] = {}
    for split in SPLITS:
        ds = PneumoniaMNIST(split=split, download=download, root=root)
        imgs = np.asarray(ds.imgs)
        if imgs.ndim == 4:  # (N,H,W,1) in some versions
            imgs = imgs[..., 0]
        x = torch.from_numpy(imgs).float().div(255.0).unsqueeze(1).contiguous()
        y = torch.from_numpy(np.asarray(ds.labels).reshape(-1)).long()
        out[split] = SplitData(x, y)
    return out


def to_gan_space(x01: torch.Tensor, img_size: int = 64) -> torch.Tensor:
    """[0,1] 28x28 -> [-1,1] img_size x img_size (bilinear upsampling)."""
    x = x01
    if x.shape[-1] != img_size:
        x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)
    return x * 2.0 - 1.0


def from_gan_space(x: torch.Tensor, out_size: int = 28) -> torch.Tensor:
    """[-1,1] GAN space -> [0,1] at out_size (area downsampling = anti-aliased)."""
    x01 = (x.clamp(-1, 1) + 1.0) / 2.0
    if x01.shape[-1] != out_size:
        x01 = F.interpolate(x01, size=(out_size, out_size), mode="area")
    return x01


def roundtrip(x01: torch.Tensor, img_size: int = 64) -> torch.Tensor:
    if img_size == x01.shape[-1]:
        return x01.clone()
    return from_gan_space(to_gan_space(x01, img_size), x01.shape[-1])


def class_counts(y: torch.Tensor, n_classes: int = 2) -> list[int]:
    return [int((y == c).sum()) for c in range(n_classes)]


def dataset_summary(splits: dict[str, SplitData]) -> dict:
    mapping = {0: "normal", 1: "pneumonia"}
    try:
        mapping = get_label_mapping()
    except Exception:  # pragma: no cover - medmnist missing in some unit tests
        pass
    summary = {"label_mapping": mapping, "splits": {}}
    for name, s in splits.items():
        counts = class_counts(s.y)
        summary["splits"][name] = {
            "n": len(s),
            "counts": {mapping[c]: counts[c] for c in range(2)},
            "fraction_normal": counts[0] / max(1, len(s)),
            "imbalance_ratio_pneumonia_to_normal": counts[1] / max(1, counts[0]),
            "shape": list(s.x28.shape[1:]),
            "pixel_min": float(s.x28.min()),
            "pixel_max": float(s.x28.max()),
            "pixel_mean": float(s.x28.mean()),
            "pixel_std": float(s.x28.std()),
        }
    return summary

In [ ]:
%%writefile gan_lab/models.py
"""Conditional DCGAN generator / projection discriminator, plus a small CNN classifier.

Design choices (justified in the report):
* Generator G(z, y): the class label is embedded (nn.Embedding) and concatenated with z,
  then upsampled with the DCGAN stack ConvT-BN-ReLU and a Tanh output (pairs with the
  [-1,1] input normalisation).
* Discriminator D(x, y): DCGAN conv stack + *projection* conditioning (Miyato & Koyama,
  2018): logit = w^T h + <embed(y), h>. Projection gives D a class-specific direction in
  feature space instead of just concatenating a label map, and is the standard partner
  for spectral normalisation. Spectral norm is optional (searched by Ray Tune); when it
  is on, BatchNorm is removed from D.
"""
from __future__ import annotations

import math
from typing import Iterable

import torch
import torch.nn as nn


def _sn(module: nn.Module, enabled: bool) -> nn.Module:
    return nn.utils.spectral_norm(module) if enabled else module


def _weight(m: nn.Module) -> torch.Tensor:
    return m.weight_orig if hasattr(m, "weight_orig") else m.weight


def dcgan_init(m: nn.Module, init_embedding: bool = False) -> None:
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
        nn.init.normal_(_weight(m), 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding) and init_embedding:
        nn.init.normal_(_weight(m), 0.0, 0.02)


class Generator(nn.Module):
    def __init__(self, latent_dim: int = 64, n_classes: int = 2, ngf: int = 64,
                 img_size: int = 64, embed_dim: int = 32):
        super().__init__()
        if img_size not in (32, 64):
            raise ValueError("img_size must be 32 or 64")
        self.latent_dim, self.n_classes, self.img_size = latent_dim, n_classes, img_size
        self.embed = nn.Embedding(n_classes, embed_dim)
        n_up = int(math.log2(img_size)) - 2
        ch = ngf * 2 ** (n_up - 1)
        layers: list[nn.Module] = [
            nn.ConvTranspose2d(latent_dim + embed_dim, ch, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ch), nn.ReLU(True),
        ]
        for _ in range(n_up - 1):
            layers += [nn.ConvTranspose2d(ch, ch // 2, 4, 2, 1, bias=False),
                       nn.BatchNorm2d(ch // 2), nn.ReLU(True)]
            ch //= 2
        layers += [nn.ConvTranspose2d(ch, 1, 4, 2, 1, bias=False), nn.Tanh()]
        self.net = nn.Sequential(*layers)
        self.apply(lambda m: dcgan_init(m, init_embedding=False))

    def forward(self, z: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        h = torch.cat([z, self.embed(y)], dim=1)[:, :, None, None]
        return self.net(h)


class Discriminator(nn.Module):
    def __init__(self, n_classes: int = 2, ndf: int = 64, img_size: int = 64,
                 spectral_norm: bool = False):
        super().__init__()
        sn = bool(spectral_norm)
        n_down = int(math.log2(img_size)) - 2
        layers: list[nn.Module] = [_sn(nn.Conv2d(1, ndf, 4, 2, 1), sn), nn.LeakyReLU(0.2, True)]
        ch = ndf
        for _ in range(n_down - 1):
            layers.append(_sn(nn.Conv2d(ch, ch * 2, 4, 2, 1, bias=sn), sn))
            if not sn:
                layers.append(nn.BatchNorm2d(ch * 2))
            layers.append(nn.LeakyReLU(0.2, True))
            ch *= 2
        self.features = nn.Sequential(*layers)
        self.linear = _sn(nn.Linear(ch, 1), sn)
        self.embed = _sn(nn.Embedding(n_classes, ch), sn)
        self.apply(lambda m: dcgan_init(m, init_embedding=True))

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        h = self.features(x).sum(dim=(2, 3))  # global sum pooling -> (B, ch)
        return self.linear(h).squeeze(1) + (self.embed(y) * h).sum(dim=1)


class SmallCNN(nn.Module):
    """Small 28x28 classifier used (a) as the downstream model and (b) as the frozen
    'judge' feature extractor for Frechet distance / diversity / class consistency."""

    def __init__(self, n_classes: int = 2, width: int = 32, feat_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        w = width
        self.body = nn.Sequential(
            nn.Conv2d(1, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(True),
            nn.Conv2d(w, w, 3, padding=1), nn.BatchNorm2d(w), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(w, 2 * w, 3, padding=1), nn.BatchNorm2d(2 * w), nn.ReLU(True),
            nn.Conv2d(2 * w, 2 * w, 3, padding=1), nn.BatchNorm2d(2 * w), nn.ReLU(True),
            nn.MaxPool2d(2),
            nn.Conv2d(2 * w, 4 * w, 3, padding=1), nn.BatchNorm2d(4 * w), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.proj = nn.Sequential(nn.Linear(4 * w, feat_dim), nn.ReLU(True), nn.Dropout(dropout))
        self.head = nn.Linear(feat_dim, n_classes)

    def features(self, x01: torch.Tensor) -> torch.Tensor:
        return self.proj(self.body(x01 * 2.0 - 1.0))

    def forward(self, x01: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x01))


# ----------------------------------------------------------------------------- helpers

def build_gan(cfg: dict) -> tuple[Generator, Discriminator]:
    G = Generator(latent_dim=cfg["latent_dim"], ngf=cfg["ngf"], img_size=cfg["img_size"],
                  embed_dim=cfg.get("embed_dim", 32))
    D = Discriminator(ndf=cfg["ndf"], img_size=cfg["img_size"], spectral_norm=cfg["spectral_norm"])
    return G, D


def sample_latent(n: int, latent_dim: int, seed: int, device="cpu") -> torch.Tensor:
    """Latents drawn from a seeded *CPU* generator so a seed gives the same z on any device."""
    g = torch.Generator().manual_seed(int(seed))
    return torch.randn(n, latent_dim, generator=g).to(device)


@torch.no_grad()
def generate(G: Generator, labels: Iterable[int] | torch.Tensor, seed: int, device=None,
             batch_size: int = 512) -> torch.Tensor:
    """Generate images in GAN space ([-1,1]) for the given labels; returns a CPU tensor."""
    device = device or next(G.parameters()).device
    labels = torch.as_tensor(list(labels) if not torch.is_tensor(labels) else labels, dtype=torch.long)
    z = sample_latent(len(labels), G.latent_dim, seed, "cpu")
    out = []
    for i in range(0, len(labels), batch_size):
        out.append(G(z[i:i + batch_size].to(device), labels[i:i + batch_size].to(device)).cpu())
    return torch.cat(out) if out else torch.empty(0, 1, G.img_size, G.img_size)


def save_generator(path: str, G: Generator, cfg: dict, meta: dict | None = None) -> None:
    torch.save({"config": cfg, "G": G.state_dict(), "meta": meta or {}}, path)


def load_generator(path: str, device="cpu") -> tuple[Generator, dict, dict]:
    state = torch.load(path, map_location="cpu", weights_only=False)
    cfg = state["config"]
    G = Generator(latent_dim=cfg["latent_dim"], ngf=cfg["ngf"], img_size=cfg["img_size"],
                  embed_dim=cfg.get("embed_dim", 32))
    G.load_state_dict(state["G"])
    return G.to(device).eval(), cfg, state.get("meta", {})


def load_judge(path_or_state, device="cpu") -> SmallCNN:
    """Load the frozen real-data judge from a path or from an already-loaded state dict."""
    if isinstance(path_or_state, str):
        path_or_state = torch.load(path_or_state, map_location="cpu", weights_only=False)["state_dict"]
    m = SmallCNN()
    m.load_state_dict(path_or_state)
    return m.to(device).eval()


def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

In [ ]:
%%writefile gan_lab/trainer.py
"""Adversarial training loop for the conditional DCGAN (shared by the baseline script and
the Ray Tune trainable)."""
from __future__ import annotations

import torch
import torch.nn as nn

from .models import build_gan, count_params

DEFAULT_CONFIG = {
    "img_size": 64,
    "latent_dim": 64,
    "embed_dim": 32,
    "ngf": 64,
    "ndf": 64,
    "lr_g": 2e-4,
    "lr_d": 2e-4,
    "beta1": 0.5,
    "beta2": 0.999,
    "batch_size": 64,
    "label_smoothing": 0.1,   # one-sided: real target = 1 - label_smoothing
    "d_steps": 1,             # discriminator updates per generator update
    "spectral_norm": False,
    "seed": 0,
}


class GANTrainer:
    def __init__(self, cfg: dict, device):
        self.cfg = {**DEFAULT_CONFIG, **cfg}
        self.device = torch.device(device)
        self.G, self.D = build_gan(self.cfg)
        self.G.to(self.device)
        self.D.to(self.device)
        betas = (self.cfg["beta1"], self.cfg["beta2"])
        self.opt_g = torch.optim.Adam(self.G.parameters(), lr=self.cfg["lr_g"], betas=betas)
        self.opt_d = torch.optim.Adam(self.D.parameters(), lr=self.cfg["lr_d"], betas=betas)
        self.bce = nn.BCEWithLogitsLoss()
        self.epoch = 0
        self.step = 0

    def param_counts(self) -> dict:
        return {"G": count_params(self.G), "D": count_params(self.D)}

    def train_epoch(self, x: torch.Tensor, y: torch.Tensor) -> dict:
        """One pass over (x, y) which must already be in GAN space and on self.device."""
        cfg, G, D = self.cfg, self.G, self.D
        G.train()
        D.train()
        bs, n = int(cfg["batch_size"]), len(y)
        real_t = 1.0 - float(cfg["label_smoothing"])
        perm = torch.randperm(n, device=x.device)
        sums = {"d_loss": 0.0, "g_loss": 0.0, "d_acc": 0.0, "d_acc_real": 0.0, "d_acc_fake": 0.0,
                "d_real": 0.0, "d_fake": 0.0, "g_grad_norm": 0.0}
        n_batches = 0
        for i in range(0, n - bs + 1, bs):  # drop_last keeps BN statistics well-defined
            idx = perm[i:i + bs]
            real, lab = x[idx], y[idx]
            # ---- discriminator: maximise log D(x|y) + log(1 - D(G(z|y)|y))
            for _ in range(int(cfg["d_steps"])):
                z = torch.randn(bs, G.latent_dim, device=x.device)
                fake = G(z, lab)
                real_logits = D(real, lab)
                fake_logits = D(fake.detach(), lab)  # detach: no generator gradients here
                d_loss = (self.bce(real_logits, torch.full_like(real_logits, real_t))
                          + self.bce(fake_logits, torch.zeros_like(fake_logits)))
                self.opt_d.zero_grad(set_to_none=True)
                d_loss.backward()
                self.opt_d.step()
            # ---- generator: non-saturating loss, maximise log D(G(z|y)|y)
            z = torch.randn(bs, G.latent_dim, device=x.device)
            fake = G(z, lab)
            g_logits = D(fake, lab)
            g_loss = self.bce(g_logits, torch.ones_like(g_logits))
            self.opt_g.zero_grad(set_to_none=True)
            g_loss.backward()
            gn = torch.sqrt(sum((p.grad.detach() ** 2).sum() for p in G.parameters() if p.grad is not None))
            self.opt_g.step()

            with torch.no_grad():
                acc_r = (real_logits > 0).float().mean().item()
                acc_f = (fake_logits < 0).float().mean().item()
                sums["d_loss"] += d_loss.item()
                sums["g_loss"] += g_loss.item()
                sums["d_acc_real"] += acc_r
                sums["d_acc_fake"] += acc_f
                sums["d_acc"] += (acc_r + acc_f) / 2
                sums["d_real"] += torch.sigmoid(real_logits).mean().item()
                sums["d_fake"] += torch.sigmoid(fake_logits).mean().item()
                sums["g_grad_norm"] += gn.item()
            n_batches += 1
            self.step += 1
        self.epoch += 1
        return {k: v / max(1, n_batches) for k, v in sums.items()}

    # ----------------------------------------------------------------------- state
    def state_dict(self) -> dict:
        return {"config": self.cfg, "G": self.G.state_dict(), "D": self.D.state_dict(),
                "opt_g": self.opt_g.state_dict(), "opt_d": self.opt_d.state_dict(),
                "epoch": self.epoch, "step": self.step}

    def load_state_dict(self, s: dict) -> None:
        self.G.load_state_dict(s["G"])
        self.D.load_state_dict(s["D"])
        self.opt_g.load_state_dict(s["opt_g"])
        self.opt_d.load_state_dict(s["opt_d"])
        self.epoch, self.step = s.get("epoch", 0), s.get("step", 0)

In [ ]:
%%writefile gan_lab/classifier.py
"""Downstream classifier training / evaluation (the task-grounded test of the GAN)."""
from __future__ import annotations

import copy
import math

import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, roc_auc_score)

from .models import SmallCNN


def random_augment(x01: torch.Tensor) -> torch.Tensor:
    """'Naive' augmentation: small rotation/translation/scale + brightness/contrast jitter.
    Horizontal flips are deliberately NOT used: they move the heart to the wrong side and
    create anatomically implausible chest X-rays."""
    b = x01.shape[0]
    dev = x01.device
    ang = (torch.rand(b, device=dev) * 2 - 1) * math.radians(10)
    scale = 1 + (torch.rand(b, device=dev) * 2 - 1) * 0.05
    tx = (torch.rand(b, device=dev) * 2 - 1) * 0.08
    ty = (torch.rand(b, device=dev) * 2 - 1) * 0.08
    cos, sin = torch.cos(ang) / scale, torch.sin(ang) / scale
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, list(x01.shape), align_corners=False)
    x = F.grid_sample(x01, grid, padding_mode="border", align_corners=False)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    c = 1 + (torch.rand(b, 1, 1, 1, device=dev) * 2 - 1) * 0.1
    br = (torch.rand(b, 1, 1, 1, device=dev) * 2 - 1) * 0.05
    return ((x - mean) * c + mean + br).clamp(0, 1)


@torch.no_grad()
def predict_proba(model: torch.nn.Module, x01: torch.Tensor, device, batch_size: int = 2048) -> np.ndarray:
    """Returns P(pneumonia) = P(class 1)."""
    model.eval()
    out = []
    for i in range(0, len(x01), batch_size):
        logits = model(x01[i:i + batch_size].to(device))
        out.append(torch.softmax(logits.float(), 1)[:, 1].cpu())
    return torch.cat(out).numpy()


def compute_metrics(y_true, prob1, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    prob1 = np.asarray(prob1, dtype=float)
    pred = (prob1 >= threshold).astype(int)
    auc = float(roc_auc_score(y_true, prob1)) if len(np.unique(y_true)) == 2 else float("nan")
    return {
        "auc": auc,
        "f1_macro": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "f1_normal": float(f1_score(y_true, pred, pos_label=0, zero_division=0)),
        "f1_pneumonia": float(f1_score(y_true, pred, pos_label=1, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, pred)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "confusion_matrix": confusion_matrix(y_true, pred, labels=[0, 1]).tolist(),
    }


def train_classifier(x: torch.Tensor, y: torch.Tensor, device, epochs: int = 15, seed: int = 0,
                     batch_size: int = 128, lr: float = 1e-3, weight_decay: float = 1e-4,
                     class_weight=None, augment: bool = False,
                     x_val: torch.Tensor | None = None, y_val: torch.Tensor | None = None,
                     select_metric: str = "f1_macro"):
    """Train SmallCNN. If validation data is given, keep the epoch with best `select_metric`
    (model selection uses validation only; the test set is never touched here)."""
    torch.manual_seed(seed)
    model = SmallCNN().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    x, y = x.to(device), y.to(device)
    w = None if class_weight is None else torch.as_tensor(class_weight, dtype=torch.float32, device=device)
    g = torch.Generator().manual_seed(seed)
    best_score, best_state, best_epoch, history = -1.0, None, -1, []
    n = len(y)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n, generator=g).to(device)
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            if len(idx) < 2:
                continue
            xb = x[idx]
            if augment:
                xb = random_augment(xb)
            loss = F.cross_entropy(model(xb), y[idx], weight=w)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
        if x_val is not None:
            m = compute_metrics(y_val.cpu().numpy(), predict_proba(model, x_val, device))
            history.append({"epoch": ep + 1, **{k: v for k, v in m.items() if k != "confusion_matrix"}})
            if m[select_metric] > best_score:
                best_score, best_epoch = m[select_metric], ep + 1
                best_state = copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, {"best_epoch": best_epoch, "best_val": best_score, "history": history}


def inverse_frequency_weights(y: torch.Tensor, n_classes: int = 2) -> list[float]:
    counts = torch.bincount(y.cpu(), minlength=n_classes).float()
    return (counts.sum() / (n_classes * counts.clamp(min=1))).tolist()

In [ ]:
%%writefile gan_lab/metrics.py
"""GAN evaluation beyond 'pretty pictures'.

Signals (all computed on 28x28 round-trip images, see data.py):
* class_consistency_c : fraction of images generated for class c that a frozen judge
                        classifier (trained on real data only) assigns to class c.
* fd_c                : Frechet distance between judge features of real vs fake class c
                        (an FID-style score with a domain-specific feature extractor,
                        because ImageNet Inception features are meaningless for 28x28 CXR).
* div_ratio_c         : mean pairwise judge-feature distance among fakes / among reals of
                        class c. ~1 means real-like diversity; << 1 means mode collapse.
* collapse_penalty    : mean_c max(0, 1 - div_ratio_c).
* memorisation        : nearest-neighbour (pixel RMSE) distance from each fake to the
                        training set, compared with the same distance for *held-out real*
                        validation images. A fake closer to a training image than 95% of
                        real validation images are is flagged. Held-out data already has a
                        5% flag rate by construction, so penalty = excess over 5%.
* val_f1_gain         : macro-F1 on the validation split of a quick classifier trained on
                        real + synthetic-minority minus the same classifier trained on real +
                        the same number of DUPLICATED real minority images (same seeds).
                        The control matters: adding *any* minority-labelled images (even
                        noise) shifts class priors and can raise macro-F1 on an imbalanced
                        split. Comparing against random oversampling isolates the value of
                        the synthetic image *content*.
* quality_score       : val_f1_gain - 0.5 * collapse_penalty - 0.5 * memorization_penalty
"""
from __future__ import annotations

import numpy as np
import torch

from .classifier import compute_metrics, predict_proba, train_classifier
from .data import SplitData, class_counts, from_gan_space, roundtrip
from .models import generate


# ------------------------------------------------------------------ primitive metrics

@torch.no_grad()
def judge_outputs(judge, x01: torch.Tensor, device, batch_size: int = 2048):
    judge.eval()
    feats, probs = [], []
    for i in range(0, len(x01), batch_size):
        f = judge.features(x01[i:i + batch_size].to(device))
        feats.append(f.float().cpu())
        probs.append(torch.softmax(judge.head(f).float(), 1).cpu())
    return torch.cat(feats), torch.cat(probs)


def _sqrtm_psd(a: np.ndarray) -> np.ndarray:
    w, v = np.linalg.eigh((a + a.T) / 2)
    return (v * np.sqrt(np.clip(w, 0, None))) @ v.T


def frechet_distance(f1, f2) -> float:
    f1 = np.asarray(f1, dtype=np.float64)
    f2 = np.asarray(f2, dtype=np.float64)
    mu1, mu2 = f1.mean(0), f2.mean(0)
    s1, s2 = np.cov(f1, rowvar=False), np.cov(f2, rowvar=False)
    r1 = _sqrtm_psd(s1)
    m = r1 @ s2 @ r1
    tr_covmean = np.sqrt(np.clip(np.linalg.eigvalsh((m + m.T) / 2), 0, None)).sum()
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1) + np.trace(s2) - 2 * tr_covmean)


def mean_pairwise_distance(f: torch.Tensor, max_n: int = 1000, seed: int = 0) -> float:
    f = torch.as_tensor(f, dtype=torch.float32)
    if len(f) > max_n:
        idx = torch.randperm(len(f), generator=torch.Generator().manual_seed(seed))[:max_n]
        f = f[idx]
    n = len(f)
    if n < 2:
        return 0.0
    d = torch.cdist(f, f)
    return float(d.sum() / (n * (n - 1)))


@torch.no_grad()
def nn_distances(query: torch.Tensor, ref: torch.Tensor, chunk: int = 1024):
    """Nearest-neighbour distance (RMSE per pixel) and index from each query row to ref."""
    q = query.flatten(1).float()
    r = ref.flatten(1).float().to(q.device)
    dims = q.shape[1]
    dists, idxs = [], []
    for i in range(0, len(q), chunk):
        d = torch.cdist(q[i:i + chunk], r)
        m, j = d.min(dim=1)
        dists.append(m / dims ** 0.5)
        idxs.append(j)
    return torch.cat(dists).cpu(), torch.cat(idxs).cpu()


def memorization_stats(fake_nn: torch.Tensor, val_nn: torch.Tensor, q: float = 0.05) -> dict:
    thr = float(torch.quantile(val_nn, q))
    rate = float((fake_nn < thr).float().mean())
    return {
        "mem_threshold": thr,
        "mem_rate": rate,
        "memorization_penalty": float(np.clip((rate - q) / (1 - q), 0, 1)),
        "fake_nn_median": float(fake_nn.median()),
        "val_nn_median": float(val_nn.median()),
        "nn_ratio": float(fake_nn.median() / max(1e-8, float(val_nn.median()))),
    }


def quality_score(f1_gain: float, collapse_penalty: float, memorization_penalty: float) -> float:
    return float(f1_gain - 0.5 * collapse_penalty - 0.5 * memorization_penalty)


# ------------------------------------------------------------------------ evaluator

class GANEvaluator:
    """Pre-computes everything that depends only on real data so per-epoch evaluation of
    a generator is cheap enough to run inside every Ray Tune trial."""

    def __init__(self, splits: dict[str, SplitData], judge, device, img_size: int = 64,
                 n_per_class: int = 500, clf_epochs: int = 4, clf_seeds=(0, 1),
                 eval_seed: int = 1234):
        self.device = torch.device(device)
        self.judge = judge.to(self.device).eval()
        self.n = n_per_class
        self.clf_epochs = clf_epochs
        self.clf_seeds = tuple(clf_seeds)
        self.eval_seed = eval_seed
        tr, va = splits["train"], splits["val"]
        self.x_tr, self.y_tr = roundtrip(tr.x28, img_size), tr.y.clone()
        self.x_va, self.y_va = roundtrip(va.x28, img_size), va.y.clone()

        feats, _ = judge_outputs(self.judge, self.x_tr, self.device)
        self.real_feats = {c: feats[self.y_tr == c] for c in (0, 1)}
        self.real_div = {c: mean_pairwise_distance(self.real_feats[c], max_n=n_per_class) for c in (0, 1)}
        self.real_pix_div = {c: mean_pairwise_distance(self.x_tr[self.y_tr == c].flatten(1), max_n=n_per_class)
                             for c in (0, 1)}
        self.train_flat = self.x_tr.flatten(1).to(self.device)
        self.val_nn, _ = nn_distances(self.x_va.to(self.device), self.train_flat)

        counts = class_counts(self.y_tr)
        self.minority = int(np.argmin(counts))
        self.n_syn = int(max(counts) - min(counts))
        self._base_f1 = None

    # -------------------------------------------------------------------------------
    def _quick_f1(self, x_extra=None, y_extra=None, seed: int = 0) -> float:
        x, y = self.x_tr, self.y_tr
        if x_extra is not None:
            x, y = torch.cat([x, x_extra]), torch.cat([y, y_extra])
        model, _ = train_classifier(x, y, self.device, epochs=self.clf_epochs, seed=seed, batch_size=256)
        return compute_metrics(self.y_va.numpy(), predict_proba(model, self.x_va, self.device))["f1_macro"]

    @property
    def base_f1(self) -> float:
        if self._base_f1 is None:
            with torch.random.fork_rng(devices=self._rng_devices()):
                g = torch.Generator().manual_seed(self.eval_seed)
                mi = torch.nonzero(self.y_tr == self.minority).squeeze(1)
                dup = mi[torch.randint(len(mi), (self.n_syn,), generator=g)]
                self._base_f1 = float(np.mean([self._quick_f1(self.x_tr[dup], self.y_tr[dup], seed=s)
                                               for s in self.clf_seeds]))
        return self._base_f1

    def _rng_devices(self):
        if self.device.type == "cuda":
            return [self.device.index if self.device.index is not None else torch.cuda.current_device()]
        return []

    @torch.no_grad()
    def generate28(self, G, labels, seed: int) -> torch.Tensor:
        return from_gan_space(generate(G, labels, seed, self.device))

    def evaluate(self, G, downstream: bool = True) -> dict:
        was_training = G.training
        G.eval()
        try:
            with torch.random.fork_rng(devices=self._rng_devices()):
                return self._evaluate(G, downstream)
        finally:
            G.train(was_training)

    def _evaluate(self, G, downstream: bool) -> dict:
        labels = torch.tensor([0] * self.n + [1] * self.n)
        fake = self.generate28(G, labels, self.eval_seed)
        out: dict[str, float] = {}
        if not torch.isfinite(fake).all():
            return {"quality_score": -1.0, "nonfinite": 1.0}
        feats, probs = judge_outputs(self.judge, fake, self.device)
        collapse = []
        for c in (0, 1):
            m = labels == c
            out[f"class_consistency_{c}"] = float((probs[m].argmax(1) == c).float().mean())
            ref = self.real_feats[c]
            if len(ref) > 2000:
                ref = ref[torch.randperm(len(ref), generator=torch.Generator().manual_seed(0))[:2000]]
            out[f"fd_{c}"] = frechet_distance(feats[m].numpy(), ref.numpy())
            out[f"div_ratio_{c}"] = mean_pairwise_distance(feats[m]) / max(1e-8, self.real_div[c])
            out[f"pix_div_ratio_{c}"] = mean_pairwise_distance(fake[m].flatten(1)) / max(1e-8, self.real_pix_div[c])
            collapse.append(max(0.0, 1.0 - out[f"div_ratio_{c}"]))
        out["class_consistency"] = (out["class_consistency_0"] + out["class_consistency_1"]) / 2
        out["fd_mean"] = (out["fd_0"] + out["fd_1"]) / 2
        out["collapse_penalty"] = float(np.mean(collapse))
        fake_nn, _ = nn_distances(fake.to(self.device), self.train_flat)
        out.update(memorization_stats(fake_nn, self.val_nn))

        if downstream:
            syn = self.generate28(G, [self.minority] * self.n_syn, self.eval_seed + 1)
            ys = torch.full((self.n_syn,), self.minority, dtype=torch.long)
            f1s = [self._quick_f1(syn, ys, seed=s) for s in self.clf_seeds]
            out["val_f1_aug"] = float(np.mean(f1s))
            out["val_f1_base"] = self.base_f1
            out["val_f1_gain"] = out["val_f1_aug"] - out["val_f1_base"]
        else:
            out["val_f1_gain"] = 0.0
        out["quality_score"] = quality_score(out["val_f1_gain"], out["collapse_penalty"],
                                             out["memorization_penalty"])
        return out

In [ ]:
%%writefile gan_lab/plots.py
"""Matplotlib figures shared by the scripts."""
from __future__ import annotations

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

BLUE, RED, GREY, GREEN, ORANGE = "#4C78A8", "#E45756", "#9D9D9D", "#54A24B", "#F58518"


def _clean(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.25)


def plot_training_history(history: list[dict], path: str, title: str = "cGAN training") -> None:
    ep = [h["epoch"] for h in history]
    get = lambda k: [h.get(k, np.nan) for h in history]  # noqa: E731
    fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
    ax = axes[0, 0]
    ax.plot(ep, get("d_loss"), color=BLUE, label="D loss")
    ax.plot(ep, get("g_loss"), color=RED, label="G loss")
    ax.set_title("adversarial losses")
    ax.legend(frameon=False)
    ax = axes[0, 1]
    ax.plot(ep, get("d_acc_real"), color=BLUE, label="D acc real")
    ax.plot(ep, get("d_acc_fake"), color=RED, label="D acc fake")
    ax.plot(ep, get("d_real"), color=BLUE, ls="--", label="mean D(x)")
    ax.plot(ep, get("d_fake"), color=RED, ls="--", label="mean D(G(z))")
    ax.axhline(0.5, color=GREY, lw=0.8)
    ax.set_ylim(0, 1.02)
    ax.set_title("discriminator balance")
    ax.legend(frameon=False, fontsize=8)
    ax = axes[0, 2]
    ax.plot(ep, get("g_grad_norm"), color=ORANGE)
    ax.set_title("generator gradient norm")
    ax = axes[1, 0]
    ax.plot(ep, get("fd_0"), "o-", ms=3, color=BLUE, label="normal")
    ax.plot(ep, get("fd_1"), "o-", ms=3, color=RED, label="pneumonia")
    ax.set_title("Frechet distance (judge features, lower=better)")
    ax.legend(frameon=False)
    ax = axes[1, 1]
    ax.plot(ep, get("div_ratio_0"), "o-", ms=3, color=BLUE, label="diversity ratio normal")
    ax.plot(ep, get("div_ratio_1"), "o-", ms=3, color=RED, label="diversity ratio pneumonia")
    ax.plot(ep, get("class_consistency"), "o-", ms=3, color=GREEN, label="class consistency")
    ax.axhline(1.0, color=GREY, lw=0.8)
    ax.set_title("diversity / conditioning")
    ax.legend(frameon=False, fontsize=8)
    ax = axes[1, 2]
    ax.plot(ep, get("mem_rate"), "o-", ms=3, color=ORANGE, label="memorisation flag rate")
    ax.plot(ep, get("val_f1_gain"), "o-", ms=3, color=GREEN, label="val F1 gain")
    ax.plot(ep, get("quality_score"), "o-", ms=3, color="black", label="quality score")
    ax.axhline(0.05, color=GREY, lw=0.8, ls=":")
    ax.axhline(0.0, color=GREY, lw=0.8)
    ax.set_title("selection signals")
    ax.legend(frameon=False, fontsize=8)
    for a in axes.flat:
        _clean(a)
        a.set_xlabel("epoch")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)

In [ ]:
%%writefile gan_lab/ray_compat.py
"""Import shim: Ray >= 2.41 exposes report/Checkpoint/RunConfig under ray.tune;
older Ray 2.x has them under ray.train."""
try:  # new API (Ray >= 2.41)
    from ray.tune import Checkpoint, CheckpointConfig, FailureConfig, RunConfig, get_checkpoint, report
except ImportError:  # pragma: no cover - older Ray
    from ray.train import Checkpoint, CheckpointConfig, FailureConfig, RunConfig, get_checkpoint, report

__all__ = ["Checkpoint", "CheckpointConfig", "FailureConfig", "RunConfig", "get_checkpoint", "report"]

In [ ]:
%%writefile gan_lab/service.py
"""Framework-agnostic core of the synthetic-image service (validated request model,
batched generation, PNG encoding with provenance metadata). Ray Serve wraps this in
serve_app.py; unit tests exercise it without Ray."""
from __future__ import annotations

import base64
import io
import secrets
import time
from typing import List, Optional, Tuple, Union

import numpy as np
import torch
from PIL import Image, PngImagePlugin
from pydantic import BaseModel, ConfigDict, Field, field_validator

from . import LABEL_NAMES, NAME_TO_LABEL, SYNTHETIC_DISCLAIMER, __version__
from .models import load_generator, sample_latent
from .utils import file_sha256

MAX_IMAGES_PER_REQUEST = 64


class GenerateRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    label: Union[int, str] = Field(..., description="0/'normal' or 1/'pneumonia'")
    n: int = Field(8, ge=1, le=MAX_IMAGES_PER_REQUEST, description="number of images")
    seed: Optional[int] = Field(None, ge=0, le=2**31 - 1, description="optional RNG seed for reproducibility")
    size: int = Field(64, description="output resolution: 28 (dataset native) or 64 (generator native)")
    return_grid: bool = Field(False, description="also return one PNG grid of all images")

    @field_validator("label", mode="before")
    @classmethod
    def _label(cls, v):
        if isinstance(v, bool):
            raise ValueError("label must be 0, 1, 'normal' or 'pneumonia'")
        if isinstance(v, str):
            key = v.strip().lower()
            if key.isdigit():
                v = int(key)
            elif key in NAME_TO_LABEL:
                return NAME_TO_LABEL[key]
            else:
                raise ValueError("label must be 0, 1, 'normal' or 'pneumonia'")
        if not isinstance(v, int) or v not in LABEL_NAMES:
            raise ValueError("label must be 0 (normal) or 1 (pneumonia)")
        return int(v)

    @field_validator("size")
    @classmethod
    def _size(cls, v):
        if v not in (28, 64):
            raise ValueError("size must be 28 or 64")
        return v


def encode_png(img_u8: np.ndarray, label: int, seed: int, index: int, model_id: str) -> str:
    """PNG with tEXt provenance chunks so the 'synthetic' flag travels with the file."""
    meta = PngImagePlugin.PngInfo()
    meta.add_text("Synthetic", "true")
    meta.add_text("Disclaimer", SYNTHETIC_DISCLAIMER)
    meta.add_text("Generator", f"cDCGAN-PneumoniaMNIST/{model_id}")
    meta.add_text("Label", LABEL_NAMES[label])
    meta.add_text("Seed", f"{seed}:{index}")
    buf = io.BytesIO()
    Image.fromarray(np.ascontiguousarray(img_u8, dtype=np.uint8)).save(buf, format="PNG", pnginfo=meta)
    return base64.b64encode(buf.getvalue()).decode("ascii")


def make_grid_u8(imgs: np.ndarray, nrow: int = 8, pad: int = 2) -> np.ndarray:
    n, h, w = imgs.shape
    ncol = min(nrow, n)
    nr = int(np.ceil(n / ncol))
    grid = np.full((nr * (h + pad) + pad, ncol * (w + pad) + pad), 255, dtype=np.uint8)
    for i in range(n):
        r, c = divmod(i, ncol)
        grid[pad + r * (h + pad): pad + r * (h + pad) + h, pad + c * (w + pad): pad + c * (w + pad) + w] = imgs[i]
    return grid


class SyntheticImageService:
    def __init__(self, checkpoint_path: str, device: str | None = None):
        dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.device = torch.device(dev)
        self.G, self.cfg, self.meta = load_generator(checkpoint_path, self.device)
        self.model_id = file_sha256(checkpoint_path)[:12]
        self.loaded_at = time.time()
        with torch.no_grad():  # warm-up (CUDA context, cudnn autotune)
            self.G(torch.zeros(2, self.G.latent_dim, device=self.device),
                   torch.tensor([0, 1], device=self.device))

    @torch.no_grad()
    def generate_batch(self, reqs: List[Tuple[int, int, int, int]]) -> List[np.ndarray]:
        """reqs: list of (label, n, seed, size). One forward pass for the whole batch.
        The generator is in eval mode (BatchNorm uses running statistics), so each
        request's images depend only on its own (label, seed): batching with other
        requests does not change outputs."""
        zs, ys = [], []
        for label, n, seed, _ in reqs:
            zs.append(sample_latent(n, self.G.latent_dim, seed, "cpu"))
            ys.append(torch.full((n,), label, dtype=torch.long))
        z = torch.cat(zs).to(self.device)
        y = torch.cat(ys).to(self.device)
        x = self.G(z, y).float()
        out, i = [], 0
        for label, n, seed, size in reqs:
            xi = x[i:i + n]
            i += n
            if size != xi.shape[-1]:
                xi = torch.nn.functional.interpolate((xi + 1) / 2, size=(size, size), mode="area") * 2 - 1
            out.append(((xi.clamp(-1, 1) + 1) * 127.5).round().to(torch.uint8)[:, 0].cpu().numpy())
        return out

    def build_response(self, req: GenerateRequest, seed: int, imgs: np.ndarray, t_start: float,
                       replica: str | None = None) -> dict:
        resp = {
            "synthetic": True,
            "disclaimer": SYNTHETIC_DISCLAIMER,
            "label": req.label,
            "label_name": LABEL_NAMES[req.label],
            "n": req.n,
            "seed": seed,
            "image_size": req.size,
            "format": "png_base64",
            "images": [encode_png(im, req.label, seed, k, self.model_id) for k, im in enumerate(imgs)],
            "model": {"name": "conditional-DCGAN (projection D)", "id": self.model_id,
                      "package_version": __version__,
                      "selected_epoch": self.meta.get("selected_epoch", self.meta.get("epoch"))},
            "latency_ms": round((time.perf_counter() - t_start) * 1000, 2),
        }
        if replica:
            resp["replica"] = replica
        if req.return_grid:
            resp["grid"] = encode_png(make_grid_u8(imgs), req.label, seed, -1, self.model_id)
        return resp

    def handle(self, req: GenerateRequest) -> dict:
        """Synchronous single-request path (used by tests and non-Ray usage)."""
        t0 = time.perf_counter()
        seed = req.seed if req.seed is not None else secrets.randbelow(2**31 - 1)
        imgs = self.generate_batch([(req.label, req.n, seed, req.size)])[0]
        return self.build_response(req, seed, imgs, t0)

    def model_card(self) -> dict:
        return {
            "name": "Synthetic PneumoniaMNIST generator",
            "intended_use": "Education, teaching demos, and research prototyping (e.g. stress-testing pipelines).",
            "not_intended_for": ["diagnosis or triage", "clinical decision support", "training models deployed "
                                 "on patients without real-data validation", "presenting outputs as real patient images"],
            "training_data": "PneumoniaMNIST train split (MedMNIST v2; pediatric CXR, 28x28, upsampled to "
                             f"{self.cfg['img_size']}x{self.cfg['img_size']})",
            "labels": LABEL_NAMES,
            "config": self.cfg,
            "selection": {k: v for k, v in self.meta.items() if k != "metrics_at_selection"},
            "disclaimer": SYNTHETIC_DISCLAIMER,
            "max_images_per_request": MAX_IMAGES_PER_REQUEST,
        }

In [ ]:
%%writefile inspect_data.py
"""Step 1 - load and inspect PneumoniaMNIST.

python inspect_data.py --out outputs/data
"""
from __future__ import annotations

import argparse
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

from gan_lab.data import SPLITS, class_counts, dataset_summary, load_pneumoniamnist, to_gan_space
from gan_lab.utils import save_json, set_seed


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--out", default="outputs/data")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    set_seed(args.seed)

    splits = load_pneumoniamnist(args.data_root)
    summary = dataset_summary(splits)

    # The assignment's torchvision pipeline: confirm shape and [-1,1] normalisation.
    from medmnist import PneumoniaMNIST
    from torchvision import transforms
    transform = transforms.Compose([transforms.Resize((64, 64)), transforms.ToTensor(),
                                    transforms.Normalize([0.5], [0.5])])
    train_ds = PneumoniaMNIST(split="train", transform=transform, download=True,
                              root=os.path.abspath(args.data_root))
    x0, y0 = train_ds[0]
    xs = torch.stack([train_ds[i][0] for i in range(256)])
    ours = to_gan_space(splits["train"].x28[:256], 64)
    summary["torchvision_check"] = {
        "len_train": len(train_ds), "sample_shape": list(x0.shape), "sample_label": int(np.asarray(y0).item()),
        "min": float(xs.min()), "max": float(xs.max()),
        "mean_abs_diff_vs_tensor_pipeline": float((xs - ours).abs().mean()),
    }
    save_json(summary, os.path.join(args.out, "data_summary.json"))

    print(f"Label mapping: {summary['label_mapping']}")
    for s in SPLITS:
        info = summary["splits"][s]
        print(f"{s:5s}: n={info['n']:5d} counts={info['counts']} "
              f"normal fraction={info['fraction_normal']:.3f} shape={info['shape']}")
    print("torchvision check:", summary["torchvision_check"])

    # ---- class counts
    fig, ax = plt.subplots(figsize=(6, 3.5))
    w = 0.38
    xs_ = np.arange(len(SPLITS))
    c0 = [class_counts(splits[s].y)[0] for s in SPLITS]
    c1 = [class_counts(splits[s].y)[1] for s in SPLITS]
    b0 = ax.bar(xs_ - w / 2, c0, w, label="0 = normal", color="#4C78A8")
    b1 = ax.bar(xs_ + w / 2, c1, w, label="1 = pneumonia", color="#E45756")
    ax.bar_label(b0, fontsize=8)
    ax.bar_label(b1, fontsize=8)
    ax.set_xticks(xs_, SPLITS)
    ax.set_ylabel("images")
    ax.set_title("PneumoniaMNIST class counts (minority = normal)")
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, "class_counts.png"), dpi=150)
    plt.close(fig)

    # ---- 16 examples per class
    tr = splits["train"]
    fig, axes = plt.subplots(4, 8, figsize=(9, 5))
    g = torch.Generator().manual_seed(args.seed)
    for c in (0, 1):
        idx = torch.nonzero(tr.y == c).squeeze(1)
        pick = idx[torch.randperm(len(idx), generator=g)[:16]]
        for k, j in enumerate(pick):
            ax = axes[2 * c + k // 8, k % 8]
            ax.imshow(tr.x28[j, 0], cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            if k % 8 == 0:
                ax.set_ylabel(["normal", "pneumonia"][c], fontsize=9)
    fig.suptitle("Real training images (16 per class, 28x28)")
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, "examples_per_class.png"), dpi=150)
    plt.close(fig)

    # ---- mean image + intensity histogram per class
    fig, axes = plt.subplots(1, 3, figsize=(10, 3))
    for c in (0, 1):
        axes[c].imshow(tr.x28[tr.y == c].mean(0)[0], cmap="gray", vmin=0, vmax=1)
        axes[c].set_title(f"mean image: {['normal', 'pneumonia'][c]}")
        axes[c].axis("off")
        axes[2].hist(tr.x28[tr.y == c].flatten().numpy(), bins=50, alpha=0.6, density=True,
                     label=["normal", "pneumonia"][c])
    axes[2].set_title("pixel intensity distribution")
    axes[2].legend(frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, "class_means_hist.png"), dpi=150)
    plt.close(fig)
    print(f"Saved figures to {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_judge.py
"""Train the frozen 'judge' classifier on REAL data only.

It is used as the feature extractor for Frechet distance / diversity and as the class-
consistency checker for generated images. It never sees synthetic data.

python train_judge.py --out artifacts/judge.pt
"""
from __future__ import annotations

import argparse
import os

import torch

from gan_lab.classifier import compute_metrics, inverse_frequency_weights, predict_proba, train_classifier
from gan_lab.data import load_pneumoniamnist, roundtrip
from gan_lab.utils import get_device, save_json, set_seed


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--out", default="artifacts/judge.pt")
    ap.add_argument("--img-size", type=int, default=64)
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    set_seed(args.seed)
    device = get_device()
    splits = load_pneumoniamnist(args.data_root)
    rt = {k: (roundtrip(v.x28, args.img_size), v.y) for k, v in splits.items()}
    # Class-weighted loss + light augmentation: the judge should be balanced, not a
    # majority-class detector, otherwise "class consistency" would be biased.
    model, info = train_classifier(*rt["train"], device, epochs=args.epochs, seed=args.seed,
                                   class_weight=inverse_frequency_weights(rt["train"][1]),
                                   augment=True, x_val=rt["val"][0], y_val=rt["val"][1])
    res = {"best_epoch": info["best_epoch"]}
    for s in ("val", "test"):
        res[s] = compute_metrics(rt[s][1].numpy(), predict_proba(model, rt[s][0], device))
        print(s, {k: round(v, 4) for k, v in res[s].items() if k != "confusion_matrix"},
              "CM", res[s]["confusion_matrix"])
    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    torch.save({"state_dict": model.cpu().state_dict(), "img_size": args.img_size, "metrics": res}, args.out)
    save_json(res, os.path.splitext(args.out)[0] + "_metrics.json")
    print("saved", args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_gan.py
"""Step 2 - train a single conditional DCGAN (baseline, or any config) with full logging.

python train_gan.py --epochs 60 --out runs/baseline
python train_gan.py --epochs 40 --out runs/unstable --set lr_d=1e-3 lr_g=5e-5 label_smoothing=0 d_steps=2

Logs: TensorBoard scalars + image grids (runs/<name>/tb), history.json, curves.png,
per-epoch fixed-noise grids, checkpoint.pt (full) and generator.pt (export).
"""
from __future__ import annotations

import argparse
import ast
import os
import time

import torch

from gan_lab.data import load_pneumoniamnist, to_gan_space
from gan_lab.metrics import GANEvaluator
from gan_lab.models import generate, load_judge, save_generator
from gan_lab.plots import plot_training_history
from gan_lab.trainer import DEFAULT_CONFIG, GANTrainer
from gan_lab.utils import get_device, save_grid, save_json, set_seed


def parse_overrides(items):
    out = {}
    for it in items or []:
        k, v = it.split("=", 1)
        try:
            out[k] = ast.literal_eval(v)
        except (ValueError, SyntaxError):
            out[k] = v
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--judge", default="artifacts/judge.pt")
    ap.add_argument("--out", default="runs/baseline")
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--eval-every", type=int, default=5)
    ap.add_argument("--set", nargs="*", default=[], help="config overrides, e.g. lr_d=1e-3")
    args = ap.parse_args()

    cfg = {**DEFAULT_CONFIG, **parse_overrides(args.set)}
    set_seed(cfg["seed"])
    device = get_device()
    os.makedirs(args.out, exist_ok=True)
    save_json(cfg, os.path.join(args.out, "config.json"))

    splits = load_pneumoniamnist(args.data_root)
    x = to_gan_space(splits["train"].x28, cfg["img_size"]).to(device)
    y = splits["train"].y.to(device)

    trainer = GANTrainer(cfg, device)
    print("params:", trainer.param_counts(), "device:", device)
    evaluator = None
    if os.path.exists(args.judge):
        evaluator = GANEvaluator(splits, load_judge(args.judge, device), device, img_size=cfg["img_size"])
    else:
        print(f"[warn] judge not found at {args.judge}; only adversarial metrics will be logged")

    try:
        from torch.utils.tensorboard import SummaryWriter
        writer = SummaryWriter(os.path.join(args.out, "tb"))
    except Exception as e:  # pragma: no cover
        print("[warn] TensorBoard unavailable:", e)
        writer = None

    fixed_labels = [0] * 8 + [1] * 8
    history = []
    t0 = time.time()
    for ep in range(1, args.epochs + 1):
        stats = trainer.train_epoch(x, y)
        row = {"epoch": ep, "time_s": time.time() - t0, **stats}
        if evaluator is not None and (ep % args.eval_every == 0 or ep == args.epochs or ep == 1):
            row.update(evaluator.evaluate(trainer.G, downstream=True))
        history.append(row)
        trainer.G.eval()
        grid = generate(trainer.G, fixed_labels, seed=999, device=device)
        trainer.G.train()
        save_grid(grid, os.path.join(args.out, "grids", f"epoch_{ep:03d}.png"), nrow=8)
        if writer:
            for k, v in row.items():
                if isinstance(v, (int, float)) and k != "epoch":
                    writer.add_scalar(k, v, ep)
            writer.add_images("fixed_grid(top=normal,bottom=pneumonia)", (grid + 1) / 2, ep)
        msg = (f"ep {ep:3d} | D {stats['d_loss']:.3f} G {stats['g_loss']:.3f} "
               f"Dacc {stats['d_acc']:.2f} D(x) {stats['d_real']:.2f} D(G(z)) {stats['d_fake']:.2f}")
        if "quality_score" in row:
            msg += (f" | FD {row['fd_mean']:.2f} div {row['div_ratio_0']:.2f}/{row['div_ratio_1']:.2f} "
                    f"cons {row['class_consistency']:.2f} mem {row['mem_rate']:.3f} "
                    f"dF1 {row['val_f1_gain']:+.3f} Q {row['quality_score']:+.3f}")
        print(msg, flush=True)

    if writer:
        writer.close()
    save_json(history, os.path.join(args.out, "history.json"))
    plot_training_history(history, os.path.join(args.out, "curves.png"), title=f"cGAN training: {args.out}")
    torch.save(trainer.state_dict(), os.path.join(args.out, "checkpoint.pt"))
    save_generator(os.path.join(args.out, "generator.pt"), trainer.G, trainer.cfg,
                   {"source": args.out, "epoch": args.epochs, "final_metrics": history[-1]})
    print(f"done in {time.time() - t0:.1f}s -> {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tune_gan.py
"""Steps 4-5 - distributed hyper-parameter search for the conditional GAN with Ray Tune.

Smoke test : python tune_gan.py --num-samples 2 --max-epochs 2 --smoke
Full search: python tune_gan.py --num-samples 24 --max-epochs 40 --gpus-per-trial 0.25

Ray features used
* search space with tune.loguniform / tune.choice (reproducible random search, seeded)
* ASHAScheduler early-stops weak trials on `quality_score`
* per-trial resources via tune.with_resources (fractional GPUs -> several trials share
  one A100 without oversubscription; Ray enforces the accounting)
* tune.with_parameters ships the in-memory dataset + judge weights once via the object store
* checkpoint reporting every iteration, CheckpointConfig keeps the best 2 per trial,
  get_checkpoint() makes trials resumable after failures (FailureConfig)
* ResultGrid analysis -> CSV, curves, best config, best checkpoint export
"""
from __future__ import annotations

import argparse
import math
import os
import shutil
import tempfile
import time

import torch

from gan_lab.data import load_pneumoniamnist, to_gan_space
from gan_lab.utils import save_json


def gan_trainable(config: dict, splits=None, judge_state=None):
    """Executed inside a Ray worker; one call = one trial."""
    from gan_lab.metrics import GANEvaluator
    from gan_lab.models import generate, load_judge
    from gan_lab.ray_compat import Checkpoint, get_checkpoint, report
    from gan_lab.trainer import DEFAULT_CONFIG, GANTrainer
    from gan_lab.utils import get_device, save_grid, set_seed

    cfg = {**DEFAULT_CONFIG, **config}
    if "ndf" not in config:  # tie discriminator width to generator width
        cfg["ndf"] = cfg["ngf"]
    set_seed(int(cfg["seed"]))
    device = get_device()
    trainer = GANTrainer(cfg, device)
    evaluator = GANEvaluator(splits, load_judge(judge_state, device), device, img_size=cfg["img_size"],
                             n_per_class=cfg["eval_n"], clf_epochs=cfg["clf_epochs"],
                             clf_seeds=tuple(range(cfg["clf_seeds"])))

    ckpt = get_checkpoint()
    if ckpt:  # resume (fault tolerance / PBT-style restarts)
        with ckpt.as_directory() as d:
            trainer.load_state_dict(torch.load(os.path.join(d, "checkpoint.pt"), map_location=device,
                                               weights_only=False))

    x = to_gan_space(splits["train"].x28, cfg["img_size"]).to(device)
    y = splits["train"].y.to(device)
    epr = int(cfg["epochs_per_report"])
    t0 = time.time()
    while trainer.epoch < cfg["max_epochs"]:
        for _ in range(epr):
            stats = trainer.train_epoch(x, y)
        metrics = {**stats, **evaluator.evaluate(trainer.G, downstream=True),
                   "epoch": trainer.epoch, "time_total_s_trial": time.time() - t0}
        if not all(math.isfinite(v) for v in metrics.values() if isinstance(v, float)):
            metrics = {k: (v if isinstance(v, (int, float)) and math.isfinite(v) else -1.0)
                       for k, v in metrics.items()}
            metrics["quality_score"] = -1.0
        with tempfile.TemporaryDirectory() as tmp:
            torch.save(trainer.state_dict(), os.path.join(tmp, "checkpoint.pt"))
            trainer.G.eval()
            save_grid(generate(trainer.G, [0] * 8 + [1] * 8, seed=999, device=device),
                      os.path.join(tmp, "grid.png"), nrow=8)
            trainer.G.train()
            report(metrics, checkpoint=Checkpoint.from_directory(tmp))


class ResourceMonitor:
    """Samples Ray's cluster-wide resource accounting while Tune runs (a text stand-in for
    the dashboard when port 8265 is not reachable, e.g. on Colab)."""

    def __init__(self, path: str, every_s: float = 10.0):
        import threading

        self.path, self.every_s = path, every_s
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)

    def _run(self):
        import ray

        t0 = time.time()
        with open(self.path, "w", encoding="utf-8") as f:
            f.write("t_s,cpu_total,cpu_used,gpu_total,gpu_used,gpu_util_pct,gpu_mem_used_mb\n")
            while not self._stop.is_set():
                tot, av = ray.cluster_resources(), ray.available_resources()
                util, mem = "", ""
                try:
                    import subprocess
                    q = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                                        "--format=csv,noheader,nounits"], capture_output=True, text=True,
                                       timeout=5).stdout.strip().splitlines()[0].split(",")
                    util, mem = q[0].strip(), q[1].strip()
                except Exception:
                    pass
                f.write(f"{time.time() - t0:.0f},{tot.get('CPU', 0)},{tot.get('CPU', 0) - av.get('CPU', 0)},"
                        f"{tot.get('GPU', 0)},{tot.get('GPU', 0) - av.get('GPU', 0)},{util},{mem}\n")
                f.flush()
                self._stop.wait(self.every_s)

    def start(self):
        self._thread.start()

    def stop(self):
        self._stop.set()
        self._thread.join(timeout=15)


SEARCH_KEYS = ["lr_g", "lr_d", "beta1", "latent_dim", "ngf", "label_smoothing", "d_steps",
               "spectral_norm", "batch_size"]


def build_search_space(args):
    from ray import tune

    space = {
        "lr_g": tune.loguniform(1e-5, 5e-4),
        "lr_d": tune.loguniform(1e-5, 5e-4),
        "beta1": tune.choice([0.0, 0.5]),
        "latent_dim": tune.choice([32, 64, 128]),
        "ngf": tune.choice([32, 64]),
        "label_smoothing": tune.choice([0.0, 0.1]),
        "d_steps": tune.choice([1, 2]),
        "spectral_norm": tune.choice([False, True]),
        "batch_size": tune.choice([64, 128]),
        # constants passed through config
        "img_size": args.img_size,
        "max_epochs": args.max_epochs,
        "epochs_per_report": args.epochs_per_report,
        "eval_n": 200 if args.smoke else 500,
        "clf_epochs": 2 if args.smoke else 4,
        "clf_seeds": 1 if args.smoke else 2,
        "seed": args.seed,
    }
    return space


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--num-samples", type=int, default=24)
    ap.add_argument("--max-epochs", type=int, default=40)
    ap.add_argument("--epochs-per-report", type=int, default=2, help="training epochs per Tune iteration")
    ap.add_argument("--grace-period", type=int, default=None, help="ASHA grace period (iterations)")
    ap.add_argument("--img-size", type=int, default=64)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--cpus-per-trial", type=float, default=2)
    ap.add_argument("--gpus-per-trial", type=float, default=None, help="default 0.25 if CUDA else 0")
    ap.add_argument("--max-concurrent", type=int, default=0)
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--judge", default="artifacts/judge.pt")
    ap.add_argument("--storage-path", default="ray_results")
    ap.add_argument("--name", default=None)
    ap.add_argument("--out", default=None, help="analysis output dir (default outputs/tune_<name>)")
    ap.add_argument("--export", default="artifacts/best_generator.pt")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--address", default=None, help="Ray cluster address (None = start local)")
    args = ap.parse_args()
    if args.smoke:
        args.epochs_per_report = 1
    name = args.name or ("cgan_smoke" if args.smoke else "cgan_search")
    out = args.out or os.path.join("outputs", f"tune_{name}")
    os.makedirs(out, exist_ok=True)

    import ray
    from ray import tune
    from ray.tune.schedulers import ASHAScheduler
    from gan_lab.ray_compat import CheckpointConfig, FailureConfig, RunConfig

    project_dir = os.path.dirname(os.path.abspath(__file__))
    if not ray.is_initialized():
        ray.init(address=args.address, ignore_reinit_error=True,
                 runtime_env={"env_vars": {"PYTHONPATH": project_dir}})
    print("Ray cluster resources:", ray.cluster_resources())

    gpus = args.gpus_per_trial
    if gpus is None:
        gpus = 0.25 if torch.cuda.is_available() else 0.0
    resources = {"cpu": args.cpus_per_trial, "gpu": gpus}
    print("Per-trial resources:", resources)

    splits = load_pneumoniamnist(args.data_root)
    if not os.path.exists(args.judge):
        raise SystemExit(f"judge checkpoint missing: {args.judge} (run train_judge.py first)")
    judge_state = torch.load(args.judge, map_location="cpu", weights_only=False)["state_dict"]

    max_t = max(1, args.max_epochs // args.epochs_per_report)
    grace = args.grace_period or (1 if args.smoke else max(1, min(3, max_t)))
    scheduler = ASHAScheduler(time_attr="training_iteration", max_t=max_t, grace_period=grace,
                              reduction_factor=2)
    try:
        from ray.tune.search import BasicVariantGenerator
        search_alg = BasicVariantGenerator(random_state=args.seed)
    except TypeError:  # very old Ray
        search_alg = None

    trainable = tune.with_resources(tune.with_parameters(gan_trainable, splits=splits, judge_state=judge_state),
                                    resources)
    tuner = tune.Tuner(
        trainable,
        param_space=build_search_space(args),
        tune_config=tune.TuneConfig(metric="quality_score", mode="max", scheduler=scheduler,
                                    num_samples=args.num_samples, search_alg=search_alg,
                                    max_concurrent_trials=args.max_concurrent or None),
        run_config=RunConfig(
            name=name, storage_path=os.path.abspath(args.storage_path),
            checkpoint_config=CheckpointConfig(num_to_keep=2, checkpoint_score_attribute="quality_score",
                                               checkpoint_score_order="max"),
            failure_config=FailureConfig(max_failures=1),
        ),
    )
    monitor = ResourceMonitor(os.path.join(out, "resource_log.csv"))
    monitor.start()
    t0 = time.time()
    try:
        results = tuner.fit()
    finally:
        monitor.stop()
    wall = time.time() - t0
    print(f"Tune finished in {wall / 60:.1f} min; errors: {results.num_errors}")
    analyse(results, args, out, wall, resources, max_t, grace)


def analyse(results, args, out, wall, resources, max_t, grace):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd

    from gan_lab.models import Generator, save_generator

    # one row per trial: last reported + best-ever quality score
    rows, curves = [], {}
    for r in results:
        if r.error or r.metrics is None:
            continue
        hist = r.metrics_dataframe
        tid = r.metrics.get("trial_id", os.path.basename(r.path))
        curves[tid] = hist
        row = {"trial_id": tid, **{k: r.config.get(k) for k in SEARCH_KEYS},
               "iterations": int(hist["training_iteration"].max()),
               "epochs": int(hist["epoch"].max()),
               "stopped_early": int(hist["training_iteration"].max()) < max_t,
               "best_quality_score": float(hist["quality_score"].max()),
               "final_quality_score": float(hist["quality_score"].iloc[-1])}
        best_i = hist["quality_score"].idxmax()
        for k in ["val_f1_gain", "collapse_penalty", "memorization_penalty", "fd_mean", "class_consistency",
                  "div_ratio_0", "div_ratio_1", "mem_rate", "d_acc"]:
            if k in hist:
                row[f"{k}@best"] = float(hist.loc[best_i, k])
        rows.append(row)
    df = pd.DataFrame(rows).sort_values("best_quality_score", ascending=False)
    df.to_csv(os.path.join(out, "trials.csv"), index=False)
    pd.concat([c.assign(trial_id=t) for t, c in curves.items()]).to_csv(os.path.join(out, "all_iterations.csv"),
                                                                       index=False)

    best = results.get_best_result(metric="quality_score", mode="max", scope="all")
    best_ckpt = best.get_best_checkpoint(metric="quality_score", mode="max")
    with best_ckpt.as_directory() as d:
        state = torch.load(os.path.join(d, "checkpoint.pt"), map_location="cpu", weights_only=False)
        if os.path.exists(os.path.join(d, "grid.png")):
            shutil.copy(os.path.join(d, "grid.png"), os.path.join(out, "best_checkpoint_grid.png"))
    cfg = state["config"]
    G = Generator(latent_dim=cfg["latent_dim"], ngf=cfg["ngf"], img_size=cfg["img_size"],
                  embed_dim=cfg.get("embed_dim", 32))
    G.load_state_dict(state["G"])
    bh = best.metrics_dataframe
    best_row = bh.loc[bh["quality_score"].idxmax()].to_dict()
    meta = {"source": "ray_tune", "experiment": os.path.basename(best.path), "trial_path": best.path,
            "selected_epoch": state["epoch"], "selection_metric": "quality_score (max over all reports)",
            "metrics_at_selection": {k: (float(v) if isinstance(v, (int, float)) else str(v))
                                     for k, v in best_row.items() if not str(k).startswith("config/")}}
    os.makedirs(os.path.dirname(args.export) or ".", exist_ok=True)
    save_generator(args.export, G, cfg, meta)
    torch.save(state, os.path.splitext(args.export)[0] + "_full_checkpoint.pt")
    summary = {"wall_clock_min": wall / 60, "num_trials": len(df), "num_errors": results.num_errors,
               "num_stopped_early": int(df["stopped_early"].sum()) if len(df) else 0,
               "asha": {"max_t": max_t, "grace_period": grace, "reduction_factor": 2,
                        "epochs_per_iteration": args.epochs_per_report},
               "resources_per_trial": resources, "best_config": {k: cfg[k] for k in cfg},
               "best_selected_epoch": state["epoch"], "best_metrics": meta["metrics_at_selection"],
               "export": args.export}
    save_json(summary, os.path.join(out, "tune_summary.json"))
    print("Best config:", {k: cfg[k] for k in SEARCH_KEYS})
    print("Best metrics:", {k: round(v, 4) for k, v in meta["metrics_at_selection"].items()
                            if isinstance(v, float)})
    print(df.head(10).to_string())

    # ---- figures
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for t, c in curves.items():
        lw, alpha = (2.5, 1.0) if t == best_row.get("trial_id") else (1.0, 0.45)
        axes[0].plot(c["epoch"], c["quality_score"], lw=lw, alpha=alpha)
        axes[1].plot(c["epoch"], c["fd_mean"], lw=lw, alpha=alpha)
        axes[2].plot(c["epoch"], c["d_acc"], lw=lw, alpha=alpha)
    axes[0].set_title("quality_score per trial (ASHA stops weak ones early)")
    axes[1].set_title("Frechet distance (judge features)")
    axes[2].set_title("discriminator accuracy")
    for a in axes:
        a.set_xlabel("epoch")
        a.grid(alpha=0.25)
        a.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "trial_curves.png"), dpi=150)
    plt.close(fig)

    keys = [k for k in SEARCH_KEYS if k in df]
    fig, axes = plt.subplots(2, math.ceil(len(keys) / 2), figsize=(15, 6))
    for ax, k in zip(axes.flat, keys):
        vals = df[k]
        if k in ("lr_g", "lr_d"):
            ax.scatter(vals, df["best_quality_score"], c=df["stopped_early"].map({True: "#9D9D9D", False: "#E45756"}))
            ax.set_xscale("log")
        else:
            cats = sorted(vals.astype(str).unique())
            for i, cat in enumerate(cats):
                sub = df[vals.astype(str) == cat]
                ax.scatter([i] * len(sub), sub["best_quality_score"],
                           c=sub["stopped_early"].map({True: "#9D9D9D", False: "#E45756"}))
            ax.set_xticks(range(len(cats)), cats)
        ax.set_title(k)
        ax.grid(alpha=0.25)
    for ax in list(axes.flat)[len(keys):]:
        ax.axis("off")
    fig.suptitle("best quality_score vs hyper-parameter (red = ran to max_t, grey = stopped by ASHA)")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "hparam_scatter.png"), dpi=150)
    plt.close(fig)
    rl = os.path.join(out, "resource_log.csv")
    if os.path.exists(rl):
        r = pd.read_csv(rl)
        if len(r):
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.step(r["t_s"] / 60, r["cpu_used"], where="post", label=f"CPUs reserved (of {r['cpu_total'].max():g})")
            ax.step(r["t_s"] / 60, r["gpu_used"] * 4, where="post",
                    label=f"GPU reserved x4 (of {r['gpu_total'].max():g} GPU)")
            ax.set_xlabel("minutes")
            ax.set_title("Ray resource reservations during the Tune run")
            ax.legend(frameon=False, fontsize=8)
            ax.grid(alpha=0.25)
            fig.tight_layout()
            fig.savefig(os.path.join(out, "resource_usage.png"), dpi=150)
            plt.close(fig)
    print("analysis saved to", out, "| best generator exported to", args.export)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile evaluate.py
"""Steps 3 + 6 - evaluate and stress-test the selected generator.

python evaluate.py --checkpoint artifacts/best_generator.pt --out outputs/eval --seeds 5

Produces
* fixed-seed grids per class, real-vs-fake panel, latent interpolation, same-z class swap
* quantitative GAN metrics (FD, diversity, class consistency, memorisation) - metrics.json
* nearest-neighbour memorisation figure + NN distance histogram
* failure cases: class-inconsistent samples and closest-to-training samples
* downstream classifier comparison (multiple seeds, held-out TEST set):
  real-only | class-weighted | oversample+naive aug | real+synthetic (0.5x, 1x gap)
  | synthetic-only (train-on-synthetic, test-on-real)
"""
from __future__ import annotations

import argparse
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats

from gan_lab import LABEL_NAMES
from gan_lab.classifier import compute_metrics, inverse_frequency_weights, predict_proba, train_classifier
from gan_lab.data import class_counts, from_gan_space, load_pneumoniamnist, roundtrip
from gan_lab.metrics import GANEvaluator, judge_outputs, nn_distances
from gan_lab.models import generate, load_generator, load_judge, sample_latent
from gan_lab.plots import BLUE, GREY, RED
from gan_lab.utils import save_grid, save_json, set_seed, get_device

METRICS = ["auc", "f1_macro", "f1_normal", "balanced_accuracy"]


def md_table(df: pd.DataFrame, floatfmt: str = "{:.4f}") -> str:
    cols = list(df.columns)
    lines = ["| " + " | ".join(str(c) for c in cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(floatfmt.format(v) if isinstance(v, (float, np.floating)) else str(v)
                                        for v in r.values) + " |")
    return "\n".join(lines)


# ------------------------------------------------------------------------- qualitative

def qualitative(G, cfg, splits, device, out):
    for c in (0, 1):
        x = generate(G, [c] * 64, seed=1234 + c, device=device)
        save_grid(x, os.path.join(out, f"grid_{LABEL_NAMES[c]}_seed{1234 + c}.png"), nrow=8)

    # real vs fake (28x28 space, same display)
    tr = splits["train"]
    g = torch.Generator().manual_seed(0)
    fig, axes = plt.subplots(4, 8, figsize=(9, 5))
    for c in (0, 1):
        idx = torch.nonzero(tr.y == c).squeeze(1)
        real = tr.x28[idx[torch.randperm(len(idx), generator=g)[:8]]]
        fake = from_gan_space(generate(G, [c] * 8, seed=77 + c, device=device))
        for k in range(8):
            axes[2 * c, k].imshow(real[k, 0], cmap="gray", vmin=0, vmax=1)
            axes[2 * c + 1, k].imshow(fake[k, 0], cmap="gray", vmin=0, vmax=1)
        axes[2 * c, 0].set_ylabel(f"REAL\n{LABEL_NAMES[c]}", fontsize=8)
        axes[2 * c + 1, 0].set_ylabel(f"SYNTHETIC\n{LABEL_NAMES[c]}", fontsize=8)
    for a in axes.flat:
        a.set_xticks([])
        a.set_yticks([])
    fig.suptitle("Real (rows 1,3) vs synthetic (rows 2,4) at 28x28")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "real_vs_synthetic.png"), dpi=150)
    plt.close(fig)

    # latent interpolation per class + same z with label swapped
    steps = 10
    z = sample_latent(4, G.latent_dim, seed=2024)
    rows, labels = [], []
    for c in (0, 1):
        for pair in ((0, 1), (2, 3)):
            a = torch.linspace(0, 1, steps)[:, None]
            zz = (1 - a) * z[pair[0]] + a * z[pair[1]]
            rows.append(zz)
            labels += [c] * steps
    zs = torch.cat(rows).to(device)
    with torch.no_grad():
        imgs = G(zs, torch.tensor(labels, device=device)).cpu()
    save_grid(imgs, os.path.join(out, "latent_interpolation.png"), nrow=steps)

    zsw = sample_latent(10, G.latent_dim, seed=31).to(device)
    with torch.no_grad():
        a = G(zsw, torch.zeros(10, dtype=torch.long, device=device)).cpu()
        b = G(zsw, torch.ones(10, dtype=torch.long, device=device)).cpu()
    save_grid(torch.cat([a, b]), os.path.join(out, "class_swap_same_z.png"), nrow=10)


# --------------------------------------------------------------- memorisation + failures

def memorisation_and_failures(G, cfg, splits, judge, ev: GANEvaluator, device, out, n=1000):
    labels = torch.tensor([0] * n + [1] * n)
    fake = ev.generate28(G, labels, seed=4321)
    fake_nn, fake_idx = nn_distances(fake.to(device), ev.train_flat)
    val_nn = ev.val_nn
    thr = float(torch.quantile(val_nn, 0.05))

    fig, ax = plt.subplots(figsize=(6, 3.5))
    bins = np.linspace(0, float(max(fake_nn.max(), val_nn.max())), 60)
    ax.hist(val_nn.numpy(), bins=bins, density=True, alpha=0.6, color=GREY, label="real val -> train")
    ax.hist(fake_nn.numpy(), bins=bins, density=True, alpha=0.6, color=RED, label="synthetic -> train")
    ax.axvline(thr, color="black", ls="--", lw=1, label="5th pct of val (flag threshold)")
    ax.set_xlabel("nearest-neighbour RMSE to training set (28x28, [0,1])")
    ax.set_ylabel("density")
    ax.legend(frameon=False, fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "nn_distance_hist.png"), dpi=150)
    plt.close(fig)

    # closest synthetic samples with their 3 nearest training images
    order = torch.argsort(fake_nn)[:8]
    trf = ev.train_flat
    fig, axes = plt.subplots(8, 4, figsize=(4.6, 9))
    for r, i in enumerate(order):
        d = torch.cdist(fake[i].flatten()[None].to(device), trf)[0]
        nn3 = torch.argsort(d)[:3].cpu()
        axes[r, 0].imshow(fake[i, 0], cmap="gray", vmin=0, vmax=1)
        axes[r, 0].set_ylabel(f"{LABEL_NAMES[int(labels[i])][:4]}\n{fake_nn[i]:.3f}", fontsize=7)
        for k, j in enumerate(nn3):
            axes[r, k + 1].imshow(ev.x_tr[j, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, 0].set_title("synthetic", fontsize=8)
    for k in range(3):
        axes[0, k + 1].set_title(f"train NN{k + 1}", fontsize=8)
    for a in axes.flat:
        a.set_xticks([])
        a.set_yticks([])
    fig.suptitle("Closest synthetic samples to training data", fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "memorization_nearest_neighbors.png"), dpi=150)
    plt.close(fig)

    # class-inconsistent samples (judge disagrees with requested label, most confident first)
    _, probs = judge_outputs(judge, fake, device)
    wrong_conf = probs[torch.arange(len(labels)), 1 - labels]
    fig, axes = plt.subplots(2, 8, figsize=(9, 2.8))
    n_wrong = {}
    for c in (0, 1):
        m = (labels == c) & (probs.argmax(1) != c)
        n_wrong[LABEL_NAMES[c]] = int(m.sum())
        idx = torch.nonzero(m).squeeze(1)
        idx = idx[torch.argsort(wrong_conf[idx], descending=True)][:8]
        for k in range(8):
            ax = axes[c, k]
            ax.set_xticks([])
            ax.set_yticks([])
            if k < len(idx):
                ax.imshow(fake[idx[k], 0], cmap="gray", vmin=0, vmax=1)
                ax.set_title(f"p={wrong_conf[idx[k]]:.2f}", fontsize=7)
        axes[c, 0].set_ylabel(f"asked:\n{LABEL_NAMES[c]}", fontsize=8)
    fig.suptitle("Failure cases: judge confidently assigns the OTHER class", fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "failure_class_inconsistent.png"), dpi=150)
    plt.close(fig)

    exact_dupes = int((fake_nn < 1e-3).sum())
    return {"n_class_inconsistent": n_wrong, "n_near_exact_duplicates": exact_dupes,
            "closest_nn_rmse": float(fake_nn.min()), "flag_threshold": thr,
            "flag_rate": float((fake_nn < thr).float().mean())}


# ------------------------------------------------------------------------- downstream

def downstream(G, splits, device, out, seeds, epochs, img_size):
    rt = {k: (roundtrip(v.x28, img_size), v.y) for k, v in splits.items()}
    xtr, ytr = rt["train"]
    counts = class_counts(ytr)
    minority = int(np.argmin(counts))
    gap = max(counts) - min(counts)
    rows, cms = [], {}

    def run(name, x, y, seed, **kw):
        model, info = train_classifier(x, y, device, epochs=epochs, seed=seed,
                                       x_val=rt["val"][0], y_val=rt["val"][1], **kw)
        m = compute_metrics(rt["test"][1].numpy(), predict_proba(model, rt["test"][0], device))
        cms.setdefault(name, np.zeros((2, 2), dtype=int))
        cms[name] += np.array(m["confusion_matrix"])
        rows.append({"condition": name, "seed": seed, "best_val_epoch": info["best_epoch"],
                     "n_train": len(y), **{k: m[k] for k in METRICS + ["f1_pneumonia", "accuracy"]}})
        print(f"  {name:22s} seed={seed} " + " ".join(f"{k}={m[k]:.3f}" for k in METRICS), flush=True)

    for seed in seeds:
        print(f"[downstream] seed {seed}")
        run("real_only", xtr, ytr, seed)
        run("real_class_weighted", xtr, ytr, seed, class_weight=inverse_frequency_weights(ytr))
        g = torch.Generator().manual_seed(seed)
        mi = torch.nonzero(ytr == minority).squeeze(1)
        extra = mi[torch.randint(len(mi), (gap,), generator=g)]
        run("real_oversample_aug", torch.cat([xtr, xtr[extra]]), torch.cat([ytr, ytr[extra]]), seed, augment=True)
        for frac in (0.5, 1.0):
            k = int(round(frac * gap))
            syn = from_gan_space(generate(G, [minority] * k, seed=10_000 + seed, device=device))
            run(f"real_plus_synth_{frac:g}x", torch.cat([xtr, syn]),
                torch.cat([ytr, torch.full((k,), minority, dtype=torch.long)]), seed)
        half = len(ytr) // 2
        syn_all = from_gan_space(generate(G, [0] * half + [1] * half, seed=20_000 + seed, device=device))
        run("synthetic_only_TSTR", syn_all, torch.tensor([0] * half + [1] * half), seed)

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, "downstream_per_seed.csv"), index=False)
    order = list(dict.fromkeys(df["condition"]))
    agg = df.groupby("condition", sort=False)[METRICS].agg(["mean", "std"])
    summary = pd.DataFrame({"condition": order})
    for mt in METRICS:
        summary[mt] = [f"{agg.loc[c, (mt, 'mean')]:.4f} ± {agg.loc[c, (mt, 'std')]:.4f}" for c in order]

    # paired comparison vs real_only (same seeds = same init + data order)
    base = df[df.condition == "real_only"].set_index("seed")
    paired = []
    for c in order:
        if c == "real_only":
            continue
        other = df[df.condition == c].set_index("seed").loc[base.index]
        for mt in METRICS:
            d = other[mt].values - base[mt].values
            n = len(d)
            se = d.std(ddof=1) / np.sqrt(n) if n > 1 else np.nan
            tcrit = stats.t.ppf(0.975, n - 1) if n > 1 else np.nan
            p = stats.ttest_rel(other[mt].values, base[mt].values).pvalue if n > 1 else np.nan
            paired.append({"condition": c, "metric": mt, "mean_delta": float(d.mean()),
                           "ci95_low": float(d.mean() - tcrit * se), "ci95_high": float(d.mean() + tcrit * se),
                           "p_value_paired_t": float(p), "wins": f"{int((d > 0).sum())}/{n}"})
    paired_df = pd.DataFrame(paired)
    paired_df.to_csv(os.path.join(out, "downstream_paired_vs_real_only.csv"), index=False)

    with open(os.path.join(out, "downstream_tables.md"), "w", encoding="utf-8") as f:
        f.write(f"### Test-set metrics (mean ± std over {len(seeds)} seeds)\n\n")
        f.write(md_table(summary) + "\n\n### Paired difference vs real_only\n\n")
        f.write(md_table(paired_df) + "\n")

    # bar chart
    fig, axes = plt.subplots(1, len(METRICS), figsize=(14, 3.8), sharey=False)
    colors = [GREY, "#BAB0AC", "#B279A2", BLUE, "#72B7B2", RED]
    for ax, mt in zip(axes, METRICS):
        means = [agg.loc[c, (mt, "mean")] for c in order]
        sds = [agg.loc[c, (mt, "std")] for c in order]
        ax.bar(range(len(order)), means, yerr=sds, color=colors[:len(order)], capsize=3)
        lo = min(m - s for m, s in zip(means, sds))
        ax.set_ylim(max(0, lo - 0.05), min(1.0, max(means) + 0.05))
        ax.set_xticks(range(len(order)), [o.replace("_", "\n") for o in order], fontsize=6)
        ax.set_title(mt)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"Downstream classifier on held-out TEST set (mean ± sd, {len(seeds)} seeds)")
    fig.tight_layout()
    fig.savefig(os.path.join(out, "downstream_bars.png"), dpi=150)
    plt.close(fig)

    # confusion matrices (summed over seeds)
    show = [c for c in ("real_only", "real_class_weighted", "real_plus_synth_1x", "synthetic_only_TSTR") if c in cms]
    fig, axes = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3))
    for ax, c in zip(np.atleast_1d(axes), show):
        cm = cms[c]
        ax.imshow(cm, cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, int(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set_xticks([0, 1], ["normal", "pneum."])
        ax.set_yticks([0, 1], ["normal", "pneum."])
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title(c, fontsize=8)
    fig.suptitle("Test confusion matrices (summed over seeds)", fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "confusion_matrices.png"), dpi=150)
    plt.close(fig)

    return {
        "summary_table": summary.to_dict(orient="records"),
        "paired": paired_df.to_dict(orient="records"),
        "means": {c: {mt: float(agg.loc[c, (mt, "mean")]) for mt in METRICS} for c in order},
        "stds": {c: {mt: float(agg.loc[c, (mt, "std")]) for mt in METRICS} for c in order},
        "confusion_matrices": {k: v.tolist() for k, v in cms.items()},
        "minority_label": minority, "gap": gap, "seeds": list(seeds), "epochs": epochs,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", default="artifacts/best_generator.pt")
    ap.add_argument("--judge", default="artifacts/judge.pt")
    ap.add_argument("--data-root", default="data")
    ap.add_argument("--out", default="outputs/eval")
    ap.add_argument("--seeds", type=int, default=5)
    ap.add_argument("--clf-epochs", type=int, default=20)
    ap.add_argument("--skip-downstream", action="store_true")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    set_seed(0)
    device = get_device()

    splits = load_pneumoniamnist(args.data_root)
    G, cfg, meta = load_generator(args.checkpoint, device)
    judge = load_judge(args.judge, device)
    print("loaded generator", cfg, "meta keys:", list(meta))

    qualitative(G, cfg, splits, device, args.out)
    ev = GANEvaluator(splits, judge, device, img_size=cfg["img_size"], n_per_class=1000,
                      clf_epochs=4, clf_seeds=(0, 1, 2))
    gan_metrics = ev.evaluate(G, downstream=True)
    print("GAN metrics:", {k: round(v, 4) for k, v in gan_metrics.items()})
    mem = memorisation_and_failures(G, cfg, splits, judge, ev, device, args.out)
    print("memorisation/failures:", mem)

    result = {"config": cfg, "checkpoint_meta": meta, "gan_metrics": gan_metrics, "memorization": mem}
    if not args.skip_downstream:
        result["downstream"] = downstream(G, splits, device, args.out, list(range(args.seeds)),
                                          args.clf_epochs, cfg["img_size"])
        print(open(os.path.join(args.out, "downstream_tables.md"), encoding="utf-8").read())
    save_json(result, os.path.join(args.out, "eval_summary.json"))
    print("saved", os.path.join(args.out, "eval_summary.json"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile regenerate_grid.py
"""Reproducible command to regenerate a fixed image grid from a saved generator.

python regenerate_grid.py --checkpoint artifacts/best_generator.pt --seed 1234 --out outputs/fixed_grid.png

Prints a SHA-256 of the uint8 images: the same checkpoint + seed + device type gives the
same hash (the latent vectors come from a seeded CPU generator, so they are identical on
every machine; tiny float differences across GPU models can still flip a pixel value).
"""
from __future__ import annotations

import argparse

import torch

from gan_lab.models import generate, load_generator
from gan_lab.utils import array_sha256, file_sha256, get_device, save_grid, to_uint8


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", default="artifacts/best_generator.pt")
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--n-per-class", type=int, default=16)
    ap.add_argument("--out", default="outputs/fixed_grid.png")
    ap.add_argument("--device", default=None)
    args = ap.parse_args()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = get_device(args.device)
    G, cfg, meta = load_generator(args.checkpoint, device)
    labels = [0] * args.n_per_class + [1] * args.n_per_class
    x = generate(G, labels, seed=args.seed, device=device)
    save_grid(x, args.out, nrow=args.n_per_class if args.n_per_class <= 16 else 16)
    print(f"checkpoint sha256[:12]={file_sha256(args.checkpoint)[:12]} seed={args.seed} device={device.type}")
    print(f"rows: first {args.n_per_class} = normal, next {args.n_per_class} = pneumonia (SYNTHETIC)")
    print(f"images sha256={array_sha256(to_uint8(x))}")
    print(f"saved {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile serve_app.py
"""Step 7 - deploy the selected generator with Ray Serve.

Python (e.g. from the notebook):
    from serve_app import deploy
    deploy("artifacts/best_generator.pt")          # -> http://127.0.0.1:8000/generate

CLI:
    CHECKPOINT=artifacts/best_generator.pt serve run serve_app:app

Endpoints
    POST /generate     {"label": 0|1|"normal"|"pneumonia", "n": 1..64, "seed": int?, "size": 28|64,
                        "return_grid": bool}
    GET  /health       liveness + model id
    GET  /model_card   intended use, limitations, config

Production concerns handled here
* validation   : pydantic model with extra="forbid", bounded n, enumerated labels -> HTTP 422
* batching     : @serve.batch merges concurrent requests into one generator forward pass
* autoscaling  : 1..4 replicas driven by ongoing requests per replica
* resources    : fractional GPU per replica (0.25 of an A100), 1 CPU
* safety       : every response has synthetic=true + disclaimer, header X-Synthetic-Data,
                 PNG tEXt provenance metadata, per-request cap; no image upload / no
                 patient data ever enters the service.
"""

import os
import secrets
import time
from typing import List, Optional, Tuple

import torch
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from ray import serve

from gan_lab import SYNTHETIC_DISCLAIMER
from gan_lab.service import GenerateRequest, SyntheticImageService

APP_NAME = "synthetic-cxr"
DEPLOYMENT_NAME = "GeneratorAPI"

api = FastAPI(title="Synthetic PneumoniaMNIST generator (education/research only)",
              description=SYNTHETIC_DISCLAIMER, version="1.0.0")


@serve.deployment(
    name=DEPLOYMENT_NAME,
    autoscaling_config={
        "min_replicas": 1,
        "initial_replicas": 1,
        "max_replicas": 4,
        "target_ongoing_requests": 4,  # scale up when replicas average > 4 in-flight requests
        "upscale_delay_s": 3,
        "downscale_delay_s": 30,
        "look_back_period_s": 5,
    },
    max_ongoing_requests=16,
    ray_actor_options={"num_cpus": 1, "num_gpus": 0},
    health_check_period_s=10,
)
@serve.ingress(api)
class GeneratorAPI:
    def __init__(self, checkpoint_path: str):
        self.svc = SyntheticImageService(checkpoint_path)
        ctx = serve.get_replica_context()
        self.replica = getattr(ctx, "replica_tag", None) or str(getattr(ctx, "replica_id", "replica"))
        self.n_requests = 0

    @serve.batch(max_batch_size=16, batch_wait_timeout_s=0.005)
    async def _generate_batched(self, items: List[Tuple[int, int, int, int]]):
        return self.svc.generate_batch(items)

    @api.post("/generate")
    async def generate(self, req: GenerateRequest):
        t0 = time.perf_counter()
        seed = req.seed if req.seed is not None else secrets.randbelow(2**31 - 1)
        imgs = await self._generate_batched((req.label, req.n, seed, req.size))
        self.n_requests += 1
        body = self.svc.build_response(req, seed, imgs, t0, replica=self.replica)
        return JSONResponse(body, headers={"X-Synthetic-Data": "true", "X-Model-Id": self.svc.model_id})

    @api.get("/health")
    async def health(self):
        return {"status": "ok", "model_id": self.svc.model_id, "device": str(self.svc.device),
                "replica": self.replica, "requests_served": self.n_requests}

    @api.get("/model_card")
    async def model_card(self):
        return self.svc.model_card()


def build_app(checkpoint_path: str, num_gpus: Optional[float] = None, min_replicas: int = 1,
              max_replicas: int = 4, target_ongoing_requests: float = 4, num_cpus: float = 1):
    if num_gpus is None:
        num_gpus = 0.25 if torch.cuda.is_available() else 0.0
    dep = GeneratorAPI.options(
        ray_actor_options={"num_cpus": num_cpus, "num_gpus": num_gpus},
        autoscaling_config={"min_replicas": min_replicas, "initial_replicas": min_replicas,
                            "max_replicas": max_replicas, "target_ongoing_requests": target_ongoing_requests,
                            "upscale_delay_s": 3, "downscale_delay_s": 30, "look_back_period_s": 5},
    )
    return dep.bind(os.path.abspath(checkpoint_path))


def deploy(checkpoint_path: str = "artifacts/best_generator.pt", host: str = "127.0.0.1", port: int = 8000,
           **kw):
    import ray

    if not ray.is_initialized():
        project_dir = os.path.dirname(os.path.abspath(__file__))
        ray.init(ignore_reinit_error=True, runtime_env={"env_vars": {"PYTHONPATH": project_dir}})
    serve.start(http_options={"host": host, "port": port})
    handle = serve.run(build_app(checkpoint_path, **kw), name=APP_NAME, route_prefix="/")
    return handle


def replica_count(app_name: str = APP_NAME, deployment: str = DEPLOYMENT_NAME) -> dict:
    """Current replica states, e.g. {'RUNNING': 2, 'STARTING': 1}."""
    st = serve.status()
    dep = st.applications[app_name].deployments[deployment]
    return {str(getattr(k, "value", k)): int(v) for k, v in dep.replica_states.items()}


# `serve run serve_app:app` entrypoint (the model is loaded lazily inside replicas)
app = build_app(os.environ.get("CHECKPOINT", "artifacts/best_generator.pt"))

In [ ]:
%%writefile client.py
"""Step 8 - example HTTP client for the Ray Serve endpoint.

python client.py --url http://127.0.0.1:8000 --out outputs/serve
"""
from __future__ import annotations

import argparse
import base64
import io
import json
import os

import requests
from PIL import Image


def decode(b64: str) -> Image.Image:
    return Image.open(io.BytesIO(base64.b64decode(b64)))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", default="http://127.0.0.1:8000")
    ap.add_argument("--out", default="outputs/serve")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    log = {}

    r = requests.get(f"{args.url}/health", timeout=30)
    print("GET /health", r.status_code, r.json())
    log["health"] = r.json()

    for label in [0, 1]:
        payload = {"label": label, "n": 8, "seed": 42, "return_grid": True}
        r = requests.post(f"{args.url}/generate", json=payload, timeout=60)
        body = r.json()
        print("POST /generate", payload, "->", r.status_code, sorted(body.keys()),
              "| X-Synthetic-Data:", r.headers.get("X-Synthetic-Data"))
        decode(body["grid"]).save(os.path.join(args.out, f"api_grid_label{label}.png"))
        img = decode(body["images"][0])
        print("   first image:", img.size, img.mode, "PNG metadata:", {k: img.info[k] for k in ("Synthetic", "Label", "Seed")})
        short = {k: v for k, v in body.items() if k not in ("images", "grid")}
        short["images"] = [body["images"][0][:60] + "...", f"... ({len(body['images'])} images)"]
        log[f"generate_label{label}"] = {"request": payload, "status": r.status_code, "response": short}

    # reproducibility: same seed -> identical bytes
    a = requests.post(f"{args.url}/generate", json={"label": "pneumonia", "n": 4, "seed": 7}, timeout=60).json()
    b = requests.post(f"{args.url}/generate", json={"label": 1, "n": 4, "seed": 7}, timeout=60).json()
    same = a["images"] == b["images"]
    print("same seed -> identical images:", same)
    log["seed_reproducible"] = same

    # validation: these must be rejected with HTTP 422
    bad = [{"label": 7, "n": 4}, {"label": "covid", "n": 4}, {"label": 0, "n": 1000},
           {"label": 0, "n": 0}, {"label": 1, "seed": -3}, {"label": 1, "size": 512},
           {"label": 0, "patient_id": "123"}]
    log["validation"] = []
    for p in bad:
        r = requests.post(f"{args.url}/generate", json=p, timeout=30)
        detail = r.json().get("detail", [{}])
        msg = detail[0].get("msg") if isinstance(detail, list) and detail else str(detail)
        print(f"invalid {json.dumps(p):40s} -> {r.status_code} {msg}")
        log["validation"].append({"request": p, "status": r.status_code, "msg": msg})

    with open(os.path.join(args.out, "client_log.json"), "w", encoding="utf-8") as f:
        json.dump(log, f, indent=2)
    print("saved", os.path.join(args.out, "client_log.json"))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile load_test.py
"""Step 8 - concurrent load test of the /generate endpoint + replica scaling trace.

python load_test.py --url http://127.0.0.1:8000 --levels 1 4 16 32 64 --duration 20

When run in the same Python process as Ray Serve (the notebook), pass
`probe=serve_app.replica_count` to record how many replicas are RUNNING over time.
"""
from __future__ import annotations

import argparse
import os
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import requests


def _worker(url, payload, stop_at, out):
    s = requests.Session()
    while time.time() < stop_at:
        t0 = time.perf_counter()
        try:
            r = s.post(url, json=payload, timeout=60)
            ok = r.status_code == 200
        except requests.RequestException:
            ok = False
        out.append((time.time(), (time.perf_counter() - t0) * 1000, ok))


def run_level(url, concurrency, duration_s, n_images):
    results = []
    stop_at = time.time() + duration_s
    payload = {"label": 0, "n": n_images}
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        for i in range(concurrency):
            p = dict(payload, label=i % 2)
            ex.submit(_worker, url, p, stop_at, results)
    lat = np.array([r[1] for r in results if r[2]])
    errors = sum(1 for r in results if not r[2])
    return {
        "concurrency": concurrency, "requests": len(results), "errors": errors,
        "throughput_rps": len(lat) / duration_s, "images_per_s": len(lat) * n_images / duration_s,
        "p50_ms": float(np.percentile(lat, 50)) if len(lat) else np.nan,
        "p95_ms": float(np.percentile(lat, 95)) if len(lat) else np.nan,
        "p99_ms": float(np.percentile(lat, 99)) if len(lat) else np.nan,
        "mean_ms": float(lat.mean()) if len(lat) else np.nan,
    }, results


def run_load_test(url="http://127.0.0.1:8000", levels=(1, 4, 16, 32, 64), duration_s=20, n_images=8,
                  probe=None, out="outputs/serve", cooldown_s=5):
    os.makedirs(out, exist_ok=True)
    endpoint = f"{url.rstrip('/')}/generate"
    requests.post(endpoint, json={"label": 0, "n": 1}, timeout=60)  # warm-up
    trace, stop = [], threading.Event()
    t_start = time.time()

    def poll():
        while not stop.is_set():
            try:
                st = probe() if probe else {}
                trace.append({"t_s": time.time() - t_start, "running": st.get("RUNNING", 0),
                              "starting": st.get("STARTING", 0)})
            except Exception as e:  # noqa: BLE001
                trace.append({"t_s": time.time() - t_start, "running": np.nan, "starting": np.nan, "err": str(e)})
            stop.wait(1.0)

    poller = threading.Thread(target=poll, daemon=True)
    if probe:
        poller.start()
    rows, phases = [], []
    try:
        for c in levels:
            t0 = time.time() - t_start
            summary, _ = run_level(endpoint, c, duration_s, n_images)
            phases.append((t0, time.time() - t_start, c))
            rows.append(summary)
            print(f"concurrency={c:3d} | {summary['throughput_rps']:7.1f} req/s | "
                  f"{summary['images_per_s']:8.1f} img/s | p50 {summary['p50_ms']:6.1f} ms | "
                  f"p95 {summary['p95_ms']:6.1f} ms | p99 {summary['p99_ms']:6.1f} ms | errors {summary['errors']}",
                  flush=True)
            time.sleep(cooldown_s)
        if probe:  # watch scale-down for a while
            time.sleep(45)
    finally:
        stop.set()
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(out, "load_test.csv"), index=False)
    tr = pd.DataFrame(trace)
    if len(tr):
        tr.to_csv(os.path.join(out, "replica_trace.csv"), index=False)
    plot(df, tr, phases, out)
    return df, tr


def plot(df, tr, phases, out):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3 if len(tr) else 2, figsize=(15 if len(tr) else 10, 3.8))
    ax = axes[0]
    ax.plot(df["concurrency"], df["p50_ms"], "o-", label="p50")
    ax.plot(df["concurrency"], df["p95_ms"], "o-", label="p95")
    ax.plot(df["concurrency"], df["p99_ms"], "o-", label="p99")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("concurrent clients")
    ax.set_ylabel("latency (ms)")
    ax.set_title("latency vs concurrency")
    ax.legend(frameon=False)
    ax = axes[1]
    ax.plot(df["concurrency"], df["throughput_rps"], "o-", color="#54A24B")
    ax.set_xscale("log", base=2)
    ax.set_xlabel("concurrent clients")
    ax.set_ylabel("requests / s")
    ax.set_title("throughput")
    if len(tr):
        ax = axes[2]
        ax.step(tr["t_s"], tr["running"], where="post", color="#E45756", label="RUNNING replicas")
        for t0, t1, c in phases:
            ax.axvspan(t0, t1, alpha=0.08, color="grey")
            ax.text((t0 + t1) / 2, 0.2, f"c={c}", ha="center", fontsize=7)
        ax.set_xlabel("seconds")
        ax.set_ylabel("replicas")
        ax.set_title("autoscaling trace")
        ax.set_ylim(0, max(4.5, float(np.nanmax(tr["running"])) + 0.5))
    for a in axes:
        a.grid(alpha=0.25)
        a.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "load_test.png"), dpi=150)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", default="http://127.0.0.1:8000")
    ap.add_argument("--levels", type=int, nargs="+", default=[1, 4, 16, 32, 64])
    ap.add_argument("--duration", type=float, default=20)
    ap.add_argument("--n-images", type=int, default=8)
    ap.add_argument("--out", default="outputs/serve")
    ap.add_argument("--probe", action="store_true", help="connect to the running Ray cluster to trace replicas")
    args = ap.parse_args()
    probe = None
    if args.probe:
        import ray

        ray.init(address="auto", ignore_reinit_error=True)
        from serve_app import replica_count
        probe = replica_count
    run_load_test(args.url, args.levels, args.duration, args.n_images, probe, args.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile make_report.py
"""Assemble the technical report (Markdown + self-contained HTML) from the experiment
outputs. Numbers and verdict sentences are filled in from the JSON/CSV results so the
report always matches the run that produced it.

python make_report.py --out outputs/report
"""
from __future__ import annotations

import argparse
import base64
import json
import os
import re

import pandas as pd

ROOT = os.path.dirname(os.path.abspath(__file__))


def jload(path, default=None):
    p = os.path.join(ROOT, path)
    if not os.path.exists(p):
        return default
    with open(p, encoding="utf-8") as f:
        return json.load(f)


def fmt(v, nd=3):
    if v is None:
        return "n/a"
    if isinstance(v, float):
        return f"{v:.{nd}f}"
    return str(v)


def dig(d, *keys, default=None):
    for k in keys:
        if not isinstance(d, dict) or k not in d:
            return default
        d = d[k]
    return d


def md_table(rows, cols, headers=None):
    headers = headers or cols
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(cols)]
    for r in rows:
        out.append("| " + " | ".join(fmt(r.get(c)) for c in cols) + " |")
    return "\n".join(out)


def fig(path, caption, width=None):
    if not os.path.exists(os.path.join(ROOT, path)):
        return f"*(figure missing: `{path}`)*"
    w = f'{{width="{width}"}}' if width else ""
    return f"![{caption}](../../{path}){w}\n\n*{caption}*"


def build(out_dir):
    data = jload("outputs/data/data_summary.json", {})
    judge = jload("artifacts/judge_metrics.json", {})
    base_hist = jload("runs/baseline/history.json", [])
    unst_hist = jload("runs/unstable/history.json", [])
    smoke = jload("outputs/tune_cgan_smoke/tune_summary.json", {})
    tune = jload("outputs/tune_cgan_search/tune_summary.json", {})
    ev = jload("outputs/eval/eval_summary.json", {})
    ev_base = jload("outputs/eval_baseline/eval_summary.json", {})
    client = jload("outputs/serve/client_log.json", {})
    repro = jload("outputs/repro.json", {})
    trials_csv = os.path.join(ROOT, "outputs/tune_cgan_search/trials.csv")
    trials = pd.read_csv(trials_csv) if os.path.exists(trials_csv) else None
    lt_csv = os.path.join(ROOT, "outputs/serve/load_test.csv")
    lt = pd.read_csv(lt_csv) if os.path.exists(lt_csv) else None
    rt_csv = os.path.join(ROOT, "outputs/serve/replica_trace.csv")
    rtrace = pd.read_csv(rt_csv) if os.path.exists(rt_csv) else None

    S = []  # report sections
    tr = dig(data, "splits", "train", default={})
    gm = ev.get("gan_metrics", {})
    mem = ev.get("memorization", {})
    ds = ev.get("downstream", {})
    means = ds.get("means", {})
    paired = pd.DataFrame(ds.get("paired", []))

    def delta(cond, metric):
        if paired.empty:
            return None
        r = paired[(paired.condition == cond) & (paired.metric == metric)]
        return None if r.empty else r.iloc[0].to_dict()

    d_syn = delta("real_plus_synth_1x", "f1_macro")
    d_cw = delta("real_class_weighted", "f1_macro")
    d_os = delta("real_oversample_aug", "f1_macro")
    d_syn_auc = delta("real_plus_synth_1x", "auc")

    def verdict(d):
        if d is None:
            return "could not be assessed (missing results)"
        lo, hi = d["ci95_low"], d["ci95_high"]
        s = f"{d['mean_delta']:+.3f} (95% CI {lo:+.3f} to {hi:+.3f}, paired t p={d['p_value_paired_t']:.3f}, wins {d['wins']})"
        if lo > 0:
            return s + " - a statistically reliable improvement"
        if hi < 0:
            return s + " - a statistically reliable degradation"
        return s + " - not distinguishable from zero with this many seeds"

    # ------------------------------------------------------------------ title + summary
    S.append("# Conditional GANs for Synthetic PneumoniaMNIST: Ray Tune search, task-grounded evaluation, and Ray Serve deployment\n")
    S.append("**DATA 255 - Homework 3a.** All generated images in this report are **SYNTHETIC**. They are "
             "not patient data and not clinical evidence. They are for education and research prototyping only.\n")
    S.append("## 1. Summary\n")
    S.append(
        "**Question.** Can a class-conditional GAN make normal and pneumonia images that are diverse enough "
        "to help a small downstream classifier under class imbalance? And how do we tell when the synthetic "
        "images are misleading?\n\n"
        f"**What we built.** We built a conditional DCGAN with a projection discriminator at 64x64. We tuned it with "
        f"Ray Tune: {fmt(tune.get('num_trials'))} trials, ASHA early stopping, and fractional-GPU trials. "
        f"We selected the model with a composite score: validation F1 gain minus collapse and memorisation penalties. "
        f"We evaluated it with qualitative grids, a domain-specific Frechet distance, diversity and memorisation checks, "
        f"and a {len(ds.get('seeds', []))}-seed downstream comparison on the held-out test set. Finally, we deployed it as "
        f"an autoscaling Ray Serve API.\n\n"
        f"**Answer.** Adding synthetic minority-class (normal) images to the training set changed test macro-F1 by "
        f"{verdict(d_syn)}, compared with training on real data only. "
        f"The simple baselines that need no generator changed it by {verdict(d_cw)} (class-weighted loss) and "
        f"{verdict(d_os)} (oversampling plus naive augmentation). The fair test for the GAN is therefore whether it "
        "beats these cheap baselines, not whether it beats doing nothing.\n")

    # ------------------------------------------------------------------ data
    S.append("## 2. Data understanding and preprocessing\n")
    rows = []
    for s in ("train", "val", "test"):
        info = dig(data, "splits", s, default={})
        c = info.get("counts", {})
        rows.append({"split": s, "n": info.get("n"), "normal (0)": c.get("normal"), "pneumonia (1)": c.get("pneumonia"),
                     "normal fraction": info.get("fraction_normal")})
    S.append(md_table(rows, ["split", "n", "normal (0)", "pneumonia (1)", "normal fraction"]))
    S.append(
        f"\n\nThe label mapping comes from the MedMNIST metadata: `{data.get('label_mapping')}`. The minority class "
        f"is **normal**. Train has a pneumonia:normal ratio of {fmt(tr.get('imbalance_ratio_pneumonia_to_normal'), 2)}:1. "
        "The test split is noticeably less imbalanced (37.5% normal vs 25.8%). This is a prior shift, so a "
        "classifier tuned to the training prior will under-call normal on test. That is one reason we report "
        "balanced accuracy, macro-F1, and AUC rather than accuracy.\n\n"
        "Preprocessing: the 28x28 uint8 images are scaled to [0,1], bilinearly resized to 64x64 (as specified in the "
        "assignment), and normalised to [-1,1], which matches the generator's Tanh output. "
        f"The assignment's torchvision `Resize->ToTensor->Normalize` pipeline gives shape "
        f"{dig(data, 'torchvision_check', 'sample_shape')} with range "
        f"[{fmt(dig(data, 'torchvision_check', 'min'), 2)}, {fmt(dig(data, 'torchvision_check', 'max'), 2)}]. "
        "Our in-memory tensor pipeline matches it to within a mean absolute difference of "
        f"{fmt(dig(data, 'torchvision_check', 'mean_abs_diff_vs_tensor_pipeline'), 4)}.\n\n"
        "*Shortcut control.* The generator outputs 64x64 images, which we downsample back to 28x28 for every "
        "classifier and memorisation check. We pass all real images through the same 28->64->28 round trip. "
        "Otherwise a classifier could learn \"slightly blurrier means synthetic, which means normal\" "
        "instead of anatomy.\n")
    S.append(fig("outputs/data/class_counts.png", "Figure 1. Class counts per split."))
    S.append(fig("outputs/data/examples_per_class.png", "Figure 2. 16 real training images per class."))
    S.append(fig("outputs/data/class_means_hist.png", "Figure 3. Per-class mean images and intensity histograms."))

    # ------------------------------------------------------------------ method
    S.append("## 3. Method\n")
    S.append(
        "**Objective.** A GAN is a two-player minimax game, "
        "min_G max_D E[log D(x|y)] + E[log(1 - D(G(z|y)|y))]. "
        "D is a binary classifier between real and fake images, given the label. G maps noise z and label y to an image. "
        "We use BCE-with-logits and the *non-saturating* generator loss (G maximises log D(G(z|y)|y)). "
        "With the original loss, G's gradients vanish exactly when D is winning. "
        "Real targets use one-sided label smoothing (1 - s) so that D does not become overconfident. During the D update, fake "
        "images are `detach()`-ed so that no gradient flows into G.\n\n"
        "**Generator G(z, y).** The label goes through a 32-d `nn.Embedding` and is concatenated with z. Then come ConvTranspose "
        "blocks 1->4->8->16->32->64 with BatchNorm+ReLU and a Tanh output. Weights use DCGAN N(0, 0.02) initialisation.\n\n"
        "**Discriminator D(x, y).** It uses strided convolutions 64->32->16->8->4 with LeakyReLU(0.2), then global sum pooling to h, and "
        "outputs a *projection* logit w^T h + embed(y)^T h (Miyato & Koyama, 2018). Projection gives D a class-specific "
        "direction in feature space, so it can penalise \"a realistic image of the wrong class\". This is stronger than "
        "concatenating a label map at the input. Spectral normalisation is optional and searched by Tune; when it is on, "
        "BatchNorm in D is removed.\n\n"
        "**Instrumentation.** Every epoch we log D and G loss, D accuracy on real and fake, mean D(x) and D(G(z)), and G "
        "gradient norm, plus a fixed-noise grid (TensorBoard + PNG). Every few epochs we also compute the evaluation "
        "metrics below.\n\n"
        "**Evaluation protocol (beyond visual quality).** A frozen *judge* CNN is trained on real data only, with "
        f"class-weighted loss (val AUC {fmt(dig(judge, 'val', 'auc'))}, test AUC {fmt(dig(judge, 'test', 'auc'))}). "
        "It provides:\n"
        "1. *Class consistency*: the fraction of images generated for class c that the judge labels c.\n"
        "2. *Frechet distance* between judge features of real and fake images, per class. This is an FID-style score with a domain-relevant "
        "feature extractor; ImageNet Inception features are not meaningful for 28x28 chest X-rays.\n"
        "3. *Diversity ratio*: mean pairwise feature distance of fakes divided by that of reals. A value near 1 means real-like spread; "
        "a value well below 1 means mode collapse. collapse_penalty = mean_c max(0, 1 - ratio_c).\n"
        "4. *Memorisation*: nearest-neighbour pixel RMSE from each fake to the training set, compared with "
        "the same distance for **held-out real validation images**. A fake is flagged if it is closer to a training "
        "image than 95% of real validation images are. Held-out data is flagged 5% of the time by construction, so "
        "memorization_penalty = excess flag rate over 5%.\n"
        "5. *Downstream utility*: macro-F1 on validation of a quick classifier trained on real + synthetic-minority data, "
        "minus the same classifier trained on real data + the same number of **duplicated real minority images**.\n\n"
        "The quality score is `quality_score = val_f1_gain - 0.5 * collapse_penalty - 0.5 * memorization_penalty`.\n\n"
        "*Why duplicated minority images as the control?* During the smoke test, a generator one epoch old, producing "
        "near-noise images, still gained +0.12 validation F1 over a real-only baseline. Adding *any* minority-labelled "
        "samples shifts the class prior and helps macro-F1 on an imbalanced split. Comparing against random oversampling "
        "isolates the contribution of the synthetic image *content*. This is an example of the misleading evaluation the assignment "
        "warns about.\n")

    # ------------------------------------------------------------------ baseline training
    S.append("## 4. Baseline training dynamics and failure modes\n")
    if base_hist:
        last = base_hist[-1]
        evals = [h for h in base_hist if "quality_score" in h]
        le = evals[-1] if evals else {}
        S.append(
            f"The baseline (default config, {last['epoch']} epochs) ended with D loss {fmt(last['d_loss'])}, "
            f"G loss {fmt(last['g_loss'])}, D accuracy {fmt(last['d_acc'])}, mean D(x) {fmt(last['d_real'])}, and mean D(G(z)) "
            f"{fmt(last['d_fake'])}. At the final evaluation: Frechet distance {fmt(le.get('fd_mean'))}, diversity ratio "
            f"{fmt(le.get('div_ratio_0'))}/{fmt(le.get('div_ratio_1'))} (normal/pneumonia), class consistency "
            f"{fmt(le.get('class_consistency'))}, memorisation flag rate {fmt(le.get('mem_rate'))}.\n")
    if unst_hist:
        ul = unst_hist[-1]
        ue = [h for h in unst_hist if "quality_score" in h]
        S.append(
            f"For contrast, we ran a deliberately **unbalanced** configuration (lr_D = 1e-3, 20x lr_G; two D steps per G step; no "
            f"label smoothing). It shows the textbook *discriminator overpowering* pattern: D accuracy {fmt(ul['d_acc'])}, "
            f"mean D(G(z)) {fmt(ul['d_fake'])}, G loss {fmt(ul['g_loss'])}, and G gradient norm {fmt(ul['g_grad_norm'])}. "
            f"Its final Frechet distance was {fmt(ue[-1].get('fd_mean') if ue else None)} and its diversity ratio "
            f"{fmt(ue[-1].get('div_ratio_0') if ue else None)}/{fmt(ue[-1].get('div_ratio_1') if ue else None)}. "
            "When D accuracy sits near 1 and D(G(z)) near 0, G gets almost no useful signal, and samples freeze or "
            "collapse. Healthy runs keep D accuracy around 0.6-0.85 and D(x) and D(G(z)) apart, but not saturated.\n")
    S.append(fig("runs/baseline/curves.png", "Figure 4. Baseline training curves (losses, D balance, metrics)."))
    S.append(fig("runs/unstable/curves.png", "Figure 5. Unbalanced configuration: the discriminator overpowers the generator."))

    # ------------------------------------------------------------------ ray tune
    S.append("## 5. Ray Tune hyper-parameter search\n")
    S.append(
        "| hyper-parameter | space | why it matters |\n|---|---|---|\n"
        "| lr_G, lr_D | loguniform(1e-5, 5e-4) each | the D/G learning-rate balance is the main lever on stability |\n"
        "| beta1 | {0.0, 0.5} | low momentum (DCGAN's 0.5, or 0) reduces oscillation in adversarial games |\n"
        "| latent_dim | {32, 64, 128} | capacity for intra-class variation (diversity vs. difficulty) |\n"
        "| ngf = ndf | {32, 64} | network width / capacity balance |\n"
        "| label_smoothing | {0.0, 0.1} | prevents an overconfident D |\n"
        "| d_steps | {1, 2} | discriminator update ratio |\n"
        "| spectral_norm (D) | {False, True} | Lipschitz control, an alternative to BatchNorm in D |\n"
        "| batch_size | {64, 128} | gradient noise and BatchNorm statistics |\n")
    asha = tune.get("asha", {})
    S.append(
        f"\n**Execution.** Random search ({fmt(tune.get('num_trials'))} trials, seeded `BasicVariantGenerator`) with "
        f"**ASHA** (`max_t`={asha.get('max_t')} iterations of {asha.get('epochs_per_iteration')} epochs, grace period "
        f"{asha.get('grace_period')}, reduction factor 2) on `quality_score`. Each trial reserves "
        f"`{tune.get('resources_per_trial')}`, so four trials share one A100 and Ray's resource accounting prevents GPU "
        "oversubscription. The dataset and judge weights are put in the object store once (`tune.with_parameters`). "
        "Every iteration reports a checkpoint (G, D, both optimisers, epoch, sample grid). `CheckpointConfig` keeps the 2 best "
        "per trial by quality_score, and `get_checkpoint()` makes trials resumable (`FailureConfig(max_failures=1)`). "
        f"Smoke test: {fmt(smoke.get('num_trials'))} trials in {fmt(smoke.get('wall_clock_min'), 1)} min. "
        f"Full search: {fmt(tune.get('wall_clock_min'), 1)} min wall-clock, with "
        f"{fmt(tune.get('num_stopped_early'))}/{fmt(tune.get('num_trials'))} trials stopped early by ASHA and "
        f"{fmt(tune.get('num_errors'))} errors.\n")
    bc = tune.get("best_config", {})
    bm = tune.get("best_metrics", {})
    S.append(
        f"**Best configuration** (selected epoch {fmt(tune.get('best_selected_epoch'))}): "
        + ", ".join(f"`{k}={fmt(bc.get(k), 6) if isinstance(bc.get(k), float) else bc.get(k)}`"
                    for k in ["lr_g", "lr_d", "beta1", "latent_dim", "ngf", "label_smoothing", "d_steps",
                              "spectral_norm", "batch_size"])
        + f". At selection: quality_score {fmt(bm.get('quality_score'))}, val F1 gain {fmt(bm.get('val_f1_gain'))}, "
          f"collapse penalty {fmt(bm.get('collapse_penalty'))}, memorisation penalty {fmt(bm.get('memorization_penalty'))}, "
          f"FD {fmt(bm.get('fd_mean'))}, class consistency {fmt(bm.get('class_consistency'))}.\n")
    if trials is not None and len(trials):
        top = trials.head(6).to_dict(orient="records")
        S.append("\nTop trials by best quality_score:\n\n" + md_table(
            top, ["lr_g", "lr_d", "beta1", "latent_dim", "ngf", "d_steps", "spectral_norm", "epochs",
                  "best_quality_score", "val_f1_gain@best", "collapse_penalty@best", "fd_mean@best"],
            ["lr_G", "lr_D", "b1", "z", "ngf", "d_steps", "SN", "epochs", "Q", "dF1", "collapse", "FD"]) + "\n")
        sn = trials.groupby("spectral_norm")["best_quality_score"].median().to_dict()
        S.append(f"\nMedian best quality_score by spectral norm: {', '.join(f'{k}: {v:.3f}' for k, v in sn.items())}. "
                 f"Median by d_steps: {', '.join(f'{k}: {v:.3f}' for k, v in trials.groupby('d_steps')['best_quality_score'].median().items())}.\n")
    S.append(fig("outputs/tune_cgan_search/trial_curves.png",
                 "Figure 6. quality_score, Frechet distance, and D accuracy per trial (best trial bold)."))
    S.append(fig("outputs/tune_cgan_search/hparam_scatter.png", "Figure 7. Best score vs each hyper-parameter."))
    S.append(fig("outputs/tune_cgan_search/resource_usage.png", "Figure 8. Ray CPU/GPU reservations during the search."))

    # ------------------------------------------------------------------ evaluation
    S.append("## 6. Evaluating and stress-testing the selected generator\n")
    S.append(md_table([
        {"metric": "class consistency (normal / pneumonia)", "value": f"{fmt(gm.get('class_consistency_0'))} / {fmt(gm.get('class_consistency_1'))}"},
        {"metric": "Frechet distance, judge features (normal / pneumonia)", "value": f"{fmt(gm.get('fd_0'))} / {fmt(gm.get('fd_1'))}"},
        {"metric": "feature diversity ratio (normal / pneumonia)", "value": f"{fmt(gm.get('div_ratio_0'))} / {fmt(gm.get('div_ratio_1'))}"},
        {"metric": "pixel diversity ratio (normal / pneumonia)", "value": f"{fmt(gm.get('pix_div_ratio_0'))} / {fmt(gm.get('pix_div_ratio_1'))}"},
        {"metric": "collapse penalty", "value": fmt(gm.get("collapse_penalty"))},
        {"metric": "memorisation flag rate (held-out real = 0.05)", "value": fmt(mem.get("flag_rate"))},
        {"metric": "median NN distance fake / real-val", "value": fmt(gm.get("nn_ratio"))},
        {"metric": "near-exact duplicates of training images (RMSE<0.001)", "value": fmt(mem.get("n_near_exact_duplicates"))},
        {"metric": "class-inconsistent samples out of 1000 per class", "value": str(mem.get("n_class_inconsistent"))},
    ], ["metric", "value"]))
    if ev_base.get("gan_metrics"):
        b = ev_base["gan_metrics"]
        S.append(f"\n\nFor comparison, the untuned baseline generator has FD {fmt(b.get('fd_mean'))}, diversity "
                 f"{fmt(b.get('div_ratio_0'))}/{fmt(b.get('div_ratio_1'))}, consistency {fmt(b.get('class_consistency'))}, "
                 f"and memorisation flag rate {fmt(b.get('mem_rate'))}.")
    S.append(
        "\n\n**Reading the memorisation check.** A flag rate near 0.05 and an NN ratio near or above 1 mean the generator is "
        "no closer to its training images than unseen real X-rays are. That is evidence against copying, but it is "
        "*not* a privacy guarantee. Pixel RMSE misses shifted or contrast-changed copies, and membership-inference "
        "attacks can succeed without near-duplicates. A flag rate well above 0.05 would be a red flag. The closest "
        "pairs are shown in Figure 12 for manual inspection.\n")
    S.append(fig(f"outputs/eval/grid_normal_seed1234.png", "Figure 9a. SYNTHETIC normal, fixed seed 1234 (64 samples)."))
    S.append(fig(f"outputs/eval/grid_pneumonia_seed1235.png", "Figure 9b. SYNTHETIC pneumonia, fixed seed 1235 (64 samples)."))
    S.append(fig("outputs/eval/real_vs_synthetic.png", "Figure 10. Real vs synthetic at the classifier's 28x28 resolution."))
    S.append(fig("outputs/eval/class_swap_same_z.png",
                 "Figure 11. Same latent z, label normal (top) vs pneumonia (bottom): what conditioning changes."))
    S.append(fig("outputs/eval/latent_interpolation.png", "Figure 11b. Latent interpolations (smooth = no memorised lookup table)."))
    S.append(fig("outputs/eval/memorization_nearest_neighbors.png", "Figure 12. Closest synthetic samples and their 3 nearest training images."))
    S.append(fig("outputs/eval/nn_distance_hist.png", "Figure 13. NN distance to train: synthetic vs held-out real."))
    S.append(fig("outputs/eval/failure_class_inconsistent.png", "Figure 14. Failure cases: the judge confidently assigns the other class."))
    S.append(
        "\n**Failure analysis.** (i) *Class leakage*: some images generated as normal carry diffuse opacities that "
        "the judge reads as pneumonia, and the reverse also happens (Figure 14). Labels on synthetic images are therefore *noisy*, which directly "
        "limits augmentation. (ii) *Low-frequency bias*: at 28x28, the GAN reproduces global shape, contrast, and lung fields well, "
        "but fine texture is smoothed. A classifier that relies on texture gains little. (iii) *Mode coverage*: diversity "
        "ratios below 1 show that the generator under-represents the tails, such as unusual patient positioning, devices, and "
        "very small children. Synthetic data cannot add variation the GAN never learned. (iv) *Metric gaming*: "
        "the composite score can be raised by shifting class priors (Section 3) or by blurry average images that lower "
        "pixel-NN distance. This is why we report the score's components separately and inspect the grids.\n")

    # ------------------------------------------------------------------ downstream
    S.append("## 7. Downstream classifier: does synthetic data help?\n")
    S.append(
        f"Protocol: the same SmallCNN, {ds.get('epochs')} epochs, Adam. Model selection uses the best validation macro-F1 epoch. "
        f"We use {len(ds.get('seeds', []))} seeds, and each seed fixes the initialisation, data order, and synthetic sample. All conditions are scored on "
        "the **same held-out test set**, which is never used for tuning. Conditions: real-only; class-weighted loss; "
        "random minority oversampling + naive augmentation (small rotation, translation, and intensity changes; no flips, because "
        "flips would put the heart on the wrong side); real + synthetic normals filling 50% or 100% of the class gap; and "
        "synthetic-only training (train-on-synthetic, test-on-real, TSTR), which measures conditional fidelity.\n")
    tbl = os.path.join(ROOT, "outputs/eval/downstream_tables.md")
    if os.path.exists(tbl):
        S.append(open(tbl, encoding="utf-8").read())
    S.append(fig("outputs/eval/downstream_bars.png", "Figure 15. Test metrics per condition (mean ± sd over seeds)."))
    S.append(fig("outputs/eval/confusion_matrices.png", "Figure 16. Test confusion matrices summed over seeds."))
    tstr = means.get("synthetic_only_TSTR", {})
    ro = means.get("real_only", {})
    S.append(
        f"\n**Interpretation.** Real + synthetic (1x gap) vs real-only, macro-F1: {verdict(d_syn)}. AUC: {verdict(d_syn_auc)}. "
        f"Class weighting: {verdict(d_cw)}. Oversampling + augmentation: {verdict(d_os)}. "
        f"A classifier trained *only* on synthetic images reaches test AUC {fmt(tstr.get('auc'))}, compared with "
        f"{fmt(ro.get('auc'))} for real-only. The GAN therefore captures much of the class-discriminative signal, but not all of it. "
        "Most of the macro-F1 movement in every 'rebalancing' condition comes from the decision threshold "
        "shifting toward the minority class; AUC, which does not depend on the threshold, moves much less. The honest conclusion is "
        "that synthetic augmentation should be judged against class weighting and oversampling, and that its benefit, "
        "if any, is small relative to seed-to-seed variance. The test-set prior shift (37.5% normal) also "
        "inflates the apparent value of any method that predicts 'normal' more often.\n")

    # ------------------------------------------------------------------ serve
    S.append("## 8. Ray Serve deployment\n")
    S.append(
        "**Endpoint contract.** `POST /generate` with JSON `{label: 0|1|\"normal\"|\"pneumonia\", n: 1..64, "
        "seed?: int >= 0, size: 28|64, return_grid?: bool}`. The response carries `synthetic: true`, a disclaimer, "
        "`label_name`, the seed used (server-chosen if omitted, so results are reproducible), base64 PNGs, the "
        "model id (checkpoint SHA-256 prefix) and selected epoch, the replica id, and latency. Every PNG embeds tEXt metadata "
        "(`Synthetic=true`, generator id, label, seed), and the HTTP response carries an `X-Synthetic-Data: true` header. "
        "`GET /health` and `GET /model_card` are also provided.\n\n"
        "**Deployment.** `@serve.deployment` with `autoscaling_config` (1-4 replicas, target 4 ongoing "
        "requests per replica, upscale delay 3 s, downscale delay 30 s), `ray_actor_options={num_cpus: 1, num_gpus: 0.25}`, "
        "`max_ongoing_requests=16`, and a FastAPI ingress. `@serve.batch(max_batch_size=16, batch_wait_timeout_s=0.005)` "
        "merges concurrent requests into one generator forward pass. Because G runs in eval mode, BatchNorm uses "
        "running statistics, so a request's images do not depend on which other requests share its batch. "
        "We unit-tested this property.\n")
    g0 = client.get("generate_label0", {})
    if g0:
        S.append("Example request and (truncated) response:\n\n```json\n" + json.dumps(g0.get("request"), indent=1)
                 + "\n" + json.dumps(g0.get("response"), indent=1)[:1500] + "\n```\n")
    if client.get("validation"):
        S.append("Input validation (pydantic, `extra=\"forbid\"`):\n\n" + md_table(
            [{"request": json.dumps(v["request"]), "status": v["status"], "message": v["msg"]} for v in client["validation"]],
            ["request", "status", "message"]) + f"\n\nSame seed gives identical images: **{client.get('seed_reproducible')}**.\n")
    S.append(fig("outputs/serve/api_grid_label0.png", "Figure 17a. Grid returned by the API (normal, seed 42), SYNTHETIC."))
    S.append(fig("outputs/serve/api_grid_label1.png", "Figure 17b. Grid returned by the API (pneumonia, seed 42), SYNTHETIC."))
    if lt is not None:
        S.append("\n**Load test** (threads posting n=8 images per request, 20 s per concurrency level):\n\n" + md_table(
            lt.to_dict(orient="records"),
            ["concurrency", "requests", "errors", "throughput_rps", "images_per_s", "p50_ms", "p95_ms", "p99_ms"]) + "\n")
        peak = lt.loc[lt["throughput_rps"].idxmax()]
        maxrep = int(rtrace["running"].max()) if rtrace is not None and len(rtrace) else None
        S.append(
            f"\nThroughput peaks at {peak['throughput_rps']:.0f} req/s ({peak['images_per_s']:.0f} images/s) at "
            f"concurrency {int(peak['concurrency'])}. The autoscaler grew the deployment to {maxrep} RUNNING replicas under load "
            "and scaled back down after the downscale delay (Figure 18). Latency at low concurrency is dominated by HTTP, proxy, and "
            "PNG/base64 encoding, not by the roughly 1 ms generator forward pass. At high concurrency, p95/p99 grow because requests "
            "queue while new replicas start; replica start-up costs several seconds (process launch, CUDA context, model load). "
            "Useful production levers: return raw uint8 or binary PNG instead of base64 JSON, raise `max_batch_size`, keep "
            "`min_replicas` >= 2 for availability, and use CPU replicas, since this model is tiny and GPU time is mostly idle.\n")
    S.append(fig("outputs/serve/load_test.png", "Figure 18. Latency, throughput, and replica count during the load test."))

    # ------------------------------------------------------------------ reproducibility
    S.append("## 9. Reproducibility\n")
    S.append(
        "* Every script takes explicit seeds. The Tune search algorithm is seeded. Latents come from a seeded **CPU** "
        "`torch.Generator`, so the same seed gives the same z on any device. Classifier seeds are paired across conditions.\n"
        "* Commands (also run end-to-end in `HW3a_ConditionalGAN.ipynb`):\n"
        "```bash\npython -m pytest -q\npython inspect_data.py\npython train_judge.py\n"
        "python train_gan.py --epochs 60 --out runs/baseline\n"
        "python tune_gan.py --num-samples 2 --max-epochs 2 --smoke\n"
        "python tune_gan.py --num-samples 24 --max-epochs 40 --gpus-per-trial 0.25\n"
        "python evaluate.py --checkpoint artifacts/best_generator.pt --seeds 5\n"
        "python regenerate_grid.py --checkpoint artifacts/best_generator.pt --seed 1234\n```\n"
        f"* Fixed-grid hash check: two independent runs of `regenerate_grid.py` produced "
        f"`{(repro.get('hash_run1') or 'n/a')[:16]}...` and `{(repro.get('hash_run2') or 'n/a')[:16]}...` "
        f"(identical: **{repro.get('identical')}**).\n"
        "* 25 unit tests cover shapes, conditioning, seeding, checkpoint round-trips, metric sanity (copies are flagged, "
        "collapse is detected), API validation, provenance metadata, and batch-invariance.\n")

    # ------------------------------------------------------------------ ethics
    S.append("## 10. Scientific reasoning, ethics, and limitations\n")
    S.append(
        "* **Synthetic is not evidence.** A GAN reproduces the statistics of its training set, including its artefacts. "
        "It cannot produce new clinical knowledge, and a plausible-looking image may show anatomy or pathology that does not "
        "exist (*hallucination*). Synthetic images must never be used as a clinical example, as a test set, or to claim "
        "diagnostic performance.\n"
        "* **Labels are noisy.** Figure 14 shows class-inconsistent samples. A synthetic 'normal' may contain disease-like "
        "patterns. Augmentation with such samples can teach a model wrong decision boundaries.\n"
        "* **Bias is inherited and amplified.** PneumoniaMNIST is derived from one paediatric cohort from a single "
        "centre (Guangzhou; Kermany et al., 2018), at 28x28 resolution, with binary labels that do not separate bacterial from viral pneumonia. "
        "There is no demographic metadata, so subgroup performance cannot be audited. Mode collapse under-samples "
        "rare presentations, which makes the synthetic set *less* diverse than reality.\n"
        "* **Privacy.** Our nearest-neighbour check found no near-copies at the pixel level, but GANs can still leak "
        "membership information. A real deployment would need differential-privacy training or formal membership-inference "
        "audits, and a data-use agreement covering synthetic derivatives.\n"
        "* **Misleading evaluation.** We found that (a) random synthetic images can raise macro-F1 by shifting class priors, "
        "(b) the test split has a different prior from train, and (c) seed variance is comparable to the effects. "
        "Attractive grids or a single run would have supported a false conclusion.\n"
        "* **Misuse.** The API could be used to fabricate 'patient' images for fraud, fake publications, or "
        "insurance claims. Mitigations: an explicit synthetic flag in every response, PNG metadata, and a header; no image upload or "
        "image-to-image editing; a per-request cap; authentication and rate limiting before any shared deployment; logging; "
        "and robust watermarking as future work. Metadata is easy to strip, so it is not a security control.\n"
        "* **Should this connect to clinical workflows?** No. At most it belongs in education (showing what pneumonia-like "
        "opacities look like, with a clear label) and in engineering stress tests (load, format, and pipeline robustness). Any "
        "model trained with synthetic data must be validated on real, external, prospectively collected data.\n\n"
        "See `report/MODEL_CARD.md` for the one-page model card.\n")

    S.append("## References\n")
    S.append(
        "1. Yang et al. *MedMNIST v2*. Scientific Data, 2023. arXiv:2110.14795.\n"
        "2. Radford, Metz, Chintala. *Unsupervised Representation Learning with DCGANs*. arXiv:1511.06434.\n"
        "3. Liaw et al. *Tune: A Research Platform for Distributed Model Selection and Training*. arXiv:1807.05118.\n"
        "4. Goodfellow et al. *Generative Adversarial Nets*. NeurIPS 2014.\n"
        "5. Mirza & Osindero. *Conditional GANs*. arXiv:1411.1784. Miyato & Koyama. *cGANs with Projection Discriminator*. ICLR 2018.\n"
        "6. Miyato et al. *Spectral Normalization for GANs*. ICLR 2018. Li et al. *Hyperband/ASHA*. MLSys 2020.\n"
        "7. Kermany et al. *Identifying Medical Diagnoses ... by Image-Based Deep Learning*. Cell, 2018.\n"
        "8. Ray documentation: Tune, Serve, autoscaling. https://docs.ray.io/en/latest/\n")

    md = "\n\n".join(S)
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "REPORT.md"), "w", encoding="utf-8") as f:
        f.write(md)
    write_html(md, os.path.join(out_dir, "REPORT.html"))
    return md


def write_html(md: str, path: str):
    def embed(m):
        alt, src = m.group(1), m.group(2)
        p = os.path.normpath(os.path.join(os.path.dirname(path), src))
        if os.path.exists(p):
            b64 = base64.b64encode(open(p, "rb").read()).decode()
            return f'<img alt="{alt}" src="data:image/png;base64,{b64}" style="max-width:100%">'
        return m.group(0)

    md2 = re.sub(r"\{width=\"[^\"]*\"\}", "", md)
    try:
        import markdown

        body = markdown.markdown(md2, extensions=["tables", "fenced_code"])
    except ImportError:  # pragma: no cover
        body = "<pre>" + md2 + "</pre>"
    body = re.sub(r'<img alt="([^"]*)" src="([^"]+)" ?/?>', embed, body)
    css = ("body{font-family:Georgia,serif;max-width:900px;margin:2em auto;padding:0 16px;line-height:1.45;color:#222}"
           "table{border-collapse:collapse;font-size:12px;margin:8px 0}td,th{border:1px solid #ccc;padding:3px 6px}"
           "code,pre{background:#f5f5f5;font-size:12px}pre{padding:8px;overflow-x:auto}img{display:block;margin:8px auto}"
           "em{color:#555}@media print{img{max-height:420px}}")
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"<!doctype html><html><head><meta charset='utf-8'><title>HW3a cGAN report</title>"
                f"<style>{css}</style></head><body>{body}</body></html>")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="outputs/report")
    a = ap.parse_args()
    build(a.out)
    print("wrote", os.path.join(a.out, "REPORT.md"), "and REPORT.html")

In [ ]:
%%writefile tests/test_models.py
import torch

from gan_lab.models import Discriminator, Generator, SmallCNN, build_gan, generate, sample_latent
from gan_lab.trainer import DEFAULT_CONFIG, GANTrainer


def test_generator_shapes_and_range():
    for size in (32, 64):
        G = Generator(latent_dim=16, ngf=8, img_size=size).eval()
        x = G(torch.randn(3, 16), torch.tensor([0, 1, 0]))
        assert x.shape == (3, 1, size, size)
        assert x.min() >= -1 and x.max() <= 1


def test_discriminator_projection_and_sn():
    for sn in (False, True):
        D = Discriminator(ndf=8, img_size=64, spectral_norm=sn)
        out = D(torch.randn(4, 1, 64, 64), torch.tensor([0, 1, 1, 0]))
        assert out.shape == (4,)
        assert torch.isfinite(out).all()


def test_label_conditions_output():
    G = Generator(latent_dim=16, ngf=8).eval()
    z = torch.randn(2, 16)
    a = G(z, torch.tensor([0, 0]))
    b = G(z, torch.tensor([1, 1]))
    assert not torch.allclose(a, b), "changing the label must change the image"


def test_seeded_generation_is_reproducible():
    G = Generator(latent_dim=16, ngf=8).eval()
    assert torch.equal(sample_latent(5, 16, 3), sample_latent(5, 16, 3))
    assert torch.equal(generate(G, [0, 1, 1], seed=11), generate(G, [0, 1, 1], seed=11))
    assert not torch.equal(generate(G, [0, 1, 1], seed=11), generate(G, [0, 1, 1], seed=12))


def test_eval_mode_output_independent_of_batch():
    """Serve batching relies on this: in eval mode a sample does not depend on its batch-mates."""
    G = Generator(latent_dim=16, ngf=8).eval()
    z = torch.randn(4, 16)
    y = torch.tensor([0, 1, 0, 1])
    with torch.no_grad():
        full = G(z, y)
        single = G(z[:1], y[:1])
    assert torch.allclose(full[:1], single, atol=1e-5)


def test_train_epoch_and_checkpoint_roundtrip():
    cfg = {**DEFAULT_CONFIG, "latent_dim": 16, "ngf": 8, "ndf": 8, "batch_size": 8}
    tr = GANTrainer(cfg, "cpu")
    x = torch.rand(32, 1, 64, 64) * 2 - 1
    y = torch.randint(0, 2, (32,))
    stats = tr.train_epoch(x, y)
    for k in ("d_loss", "g_loss", "d_acc"):
        assert torch.isfinite(torch.tensor(stats[k]))
    state = tr.state_dict()
    tr2 = GANTrainer(cfg, "cpu")
    tr2.load_state_dict(state)
    tr.G.eval()
    tr2.G.eval()
    assert torch.equal(generate(tr.G, [0, 1], 5), generate(tr2.G, [0, 1], 5))


def test_small_cnn():
    m = SmallCNN()
    assert m(torch.rand(5, 1, 28, 28)).shape == (5, 2)
    assert m.features(torch.rand(5, 1, 28, 28)).shape == (5, 128)


def test_build_gan_from_config():
    G, D = build_gan({**DEFAULT_CONFIG, "ngf": 8, "ndf": 8})
    assert G.img_size == 64

In [ ]:
%%writefile tests/test_metrics.py
import numpy as np
import torch

from gan_lab.classifier import compute_metrics, random_augment
from gan_lab.data import from_gan_space, roundtrip, to_gan_space
from gan_lab.metrics import (frechet_distance, mean_pairwise_distance, memorization_stats, nn_distances,
                             quality_score)


def _data(n=300, seed=0):
    g = torch.Generator().manual_seed(seed)
    return torch.rand(n, 1, 28, 28, generator=g)


def test_memorization_detects_copies():
    train, val = _data(400, 0), _data(100, 1)
    val_nn, _ = nn_distances(val, train)
    copies = train[:200] + 0.001 * torch.randn(200, 1, 28, 28)
    fake_nn, idx = nn_distances(copies, train)
    assert torch.equal(idx, torch.arange(200))
    assert memorization_stats(fake_nn, val_nn)["memorization_penalty"] > 0.9
    fresh_nn, _ = nn_distances(_data(200, 2), train)
    assert memorization_stats(fresh_nn, val_nn)["memorization_penalty"] < 0.1


def test_collapse_proxy():
    f = torch.randn(200, 16)
    collapsed = f[:1].repeat(200, 1)
    assert mean_pairwise_distance(collapsed) < 1e-2
    assert mean_pairwise_distance(f) > 1.0


def test_frechet_distance():
    rng = np.random.default_rng(0)
    a = rng.normal(size=(2000, 8))
    b = rng.normal(size=(2000, 8))
    assert frechet_distance(a, b) < 0.1
    assert frechet_distance(a, b + 3) > 8 * 9 * 0.9  # mean shift of 3 in 8 dims ~ 72


def test_quality_score_formula():
    assert quality_score(0.02, 0.1, 0.2) == 0.02 - 0.05 - 0.1


def test_space_conversions():
    x = _data(4)
    g = to_gan_space(x, 64)
    assert g.shape[-1] == 64 and g.min() >= -1 and g.max() <= 1
    back = from_gan_space(g, 28)
    assert back.shape == x.shape
    smooth = torch.linspace(0, 1, 28).view(1, 1, 1, 28).expand(2, 1, 28, 28).contiguous()
    assert (roundtrip(smooth, 64) - smooth).abs().mean() < 0.02  # near-lossless on smooth images


def test_compute_metrics_and_augment():
    y = np.array([0, 0, 1, 1])
    m = compute_metrics(y, np.array([0.1, 0.6, 0.7, 0.9]))
    assert m["confusion_matrix"] == [[1, 1], [0, 2]]
    assert abs(m["balanced_accuracy"] - 0.75) < 1e-9
    xa = random_augment(_data(8))
    assert xa.shape == (8, 1, 28, 28) and xa.min() >= 0 and xa.max() <= 1

In [ ]:
%%writefile tests/test_service.py
import base64
import io

import pytest
from PIL import Image
from pydantic import ValidationError

from gan_lab.models import Generator, save_generator
from gan_lab.service import GenerateRequest, SyntheticImageService
from gan_lab.trainer import DEFAULT_CONFIG


@pytest.fixture(scope="module")
def svc(tmp_path_factory):
    cfg = {**DEFAULT_CONFIG, "latent_dim": 16, "ngf": 8, "ndf": 8}
    p = tmp_path_factory.mktemp("ckpt") / "g.pt"
    G = Generator(latent_dim=16, ngf=8)
    save_generator(str(p), G, cfg, {"epoch": 1})
    return SyntheticImageService(str(p), device="cpu")


@pytest.mark.parametrize("payload", [
    {"label": 2}, {"label": "covid"}, {"label": True}, {"label": 0, "n": 0}, {"label": 0, "n": 65},
    {"label": 0, "seed": -1}, {"label": 0, "size": 128}, {"label": 0, "patient_id": "x"},
])
def test_invalid_requests_rejected(payload):
    with pytest.raises(ValidationError):
        GenerateRequest(**payload)


def test_label_aliases():
    assert GenerateRequest(label="Pneumonia").label == 1
    assert GenerateRequest(label="normal").label == 0
    assert GenerateRequest(label="1").label == 1


def test_response_is_marked_synthetic_and_reproducible(svc):
    r1 = svc.handle(GenerateRequest(label=1, n=3, seed=5, return_grid=True))
    r2 = svc.handle(GenerateRequest(label=1, n=3, seed=5))
    assert r1["synthetic"] is True and "SYNTHETIC" in r1["disclaimer"]
    assert r1["label_name"] == "pneumonia" and len(r1["images"]) == 3 and "grid" in r1
    assert r1["images"] == r2["images"]
    img = Image.open(io.BytesIO(base64.b64decode(r1["images"][0])))
    assert img.size == (64, 64) and img.info["Synthetic"] == "true"


def test_size_28_and_batching_consistency(svc):
    a = svc.generate_batch([(0, 2, 9, 28)])[0]
    b = svc.generate_batch([(1, 5, 3, 64), (0, 2, 9, 28)])[1]
    assert a.shape == (2, 28, 28)
    # identical up to float rounding: batch composition must not change a request's images
    assert abs(a.astype(int) - b.astype(int)).max() <= 1

In [ ]:
%%writefile report/MODEL_CARD.md
# Model Card: Synthetic PneumoniaMNIST Generator (cDCGAN + Ray Serve)

> **Every output of this model is SYNTHETIC.** It is not patient data and not clinical evidence. Use it only for education and research prototyping.

| | |
|---|---|
| **Model** | Class-conditional DCGAN. The generator takes a label embedding concatenated with z, followed by ConvT-BN-ReLU layers and a Tanh output. The discriminator uses projection conditioning, with spectral norm optional. Output is 64x64 grayscale; the API can return 28x28. |
| **Version / id** | `artifacts/best_generator.pt`. The model id is the first 12 characters of the checkpoint's SHA-256 and is returned by `/health` and in every response. |
| **Selection** | Chosen by Ray Tune random search with ASHA (search space in report §5). The selection score was `val_F1_gain − 0.5·collapse − 0.5·memorisation`, where F1 gain is measured against a randomly-oversampled real baseline. |
| **Training data** | PneumoniaMNIST train split (4,708 images: 1,214 normal and 3,494 pneumonia). Images are 28x28 and were upsampled to 64x64. Source: Kermany et al. (2018), a paediatric (ages 1–5) chest X-ray cohort from a single centre in Guangzhou, China. The data is distributed via MedMNIST v2 under CC BY 4.0. |
| **Labels** | `0 = normal` and `1 = pneumonia`, taken from the MedMNIST INFO metadata. Pneumonia covers both bacterial and viral cases, which are not distinguished. |

## Intended use
* Teaching: showing what GAN training, conditioning, mode collapse and memorisation look like, and illustrating normal vs. pneumonia-like appearance at low resolution, always labelled as synthetic.
* Engineering: stress testing data pipelines, serving infrastructure, autoscaling and file-format handling.
* Methods research: studying how synthetic augmentation interacts with class imbalance, always compared against class-weighting and oversampling baselines.

## Out of scope (do not use for)
* Diagnosis, triage, screening, or any clinical decision support.
* Presenting outputs as real patient images, including in publications, slides, insurance or legal contexts.
* Creating or augmenting test sets, or making claims about diagnostic accuracy.
* Training models that will be deployed on patients without validation on real, external data.
* Adult populations, other modalities or other resolutions. The model has never seen these.

## Evaluation (filled from `outputs/eval/eval_summary.json`; see report §6–7)
* **Quality and conditioning:** class consistency is judged by a classifier trained on real data only. We report Fréchet distance per class in judge-feature space and the diversity ratio relative to real images.
* **Memorisation:** we compare each sample's nearest-neighbour pixel distance to the training set with the same distance for held-out real images. The flag rate for held-out real images is 5% by construction. We also inspect the closest pairs visually.
* **Utility:** we test downstream classifiers on the held-out test set across several seeds, with paired comparisons against real-only, class-weighted and oversampling+augmentation baselines. We also report a train-on-synthetic, test-on-real score.

## Known limitations and risks
* **Hallucination and label noise:** some samples generated for one class look like the other class to the judge (report Fig. 14). Synthetic labels are not ground truth.
* **Low fidelity:** the data is 28x28 in origin. Fine texture is smoothed, and clinically relevant detail is absent.
* **Coverage and bias:** diversity is below that of the real data (mode under-coverage), so rare presentations are under-represented. The training cohort is from a single site and a single age band, with no demographic metadata, so subgroup fairness cannot be assessed.
* **Privacy:** no near-copies were found by the pixel-space check. This is **not** a formal privacy guarantee: membership inference is still possible, and the model was not trained with differential privacy.
* **Evaluation pitfalls:** rebalancing alone (even with noise images) can raise macro-F1. The test split also has a different class prior from the training split. Small effects can be within seed variance.

## Misuse risks and mitigations
| Risk | Mitigation in this prototype | Still needed for any shared deployment |
|---|---|---|
| Fabricated "patient" images (fraud, fake papers) | `synthetic: true` and a disclaimer in every response; `X-Synthetic-Data` header; PNG tEXt provenance metadata | Robust invisible watermarking, auth, rate limits, audit logging |
| Mass generation / scraping | Cap of 64 images per request; bounded autoscaling (max 4 replicas) | Per-user quotas, API keys |
| Upload of real patient data | No upload or image-to-image endpoint; `extra="forbid"` rejects unknown fields such as `patient_id` | Data-governance review |
| Over-trust by learners | `/model_card` endpoint; disclaimers | Course materials explaining the limitations |

## Serving details
`POST /generate {label, n≤64, seed?, size∈{28,64}, return_grid?}`. The service runs on Ray Serve with 1–4 autoscaling replicas (target 4 ongoing requests per replica) and 0.25 GPU plus 1 CPU per replica. `@serve.batch` (up to 16 requests, 5 ms wait) merges concurrent requests. Outputs are deterministic for a given seed and do not depend on batching. Latency and throughput figures are in report §8.

**Contact / ownership:** DATA 255 course project. This is a prototype, not maintained for production.

### 0.2 Install the package and run the unit tests (smoke tests)

In [ ]:
!pip install -q -e .
!python -m pytest -q

In [ ]:
from IPython.display import Image, Markdown, display
import json, os, pandas as pd
def show(*paths, width=None):
    for p in paths:
        if os.path.exists(p):
            display(Image(filename=p, width=width))
        else:
            print("missing:", p)

## 1. Load and inspect PneumoniaMNIST

In [ ]:
!python inspect_data.py --out outputs/data
show("outputs/data/class_counts.png", "outputs/data/examples_per_class.png", "outputs/data/class_means_hist.png")

## 3 (prerequisite). Judge classifier for the evaluation protocol
We train a frozen classifier on **real** data only. It provides the feature space for the Fréchet distance and diversity
metrics, and it checks class consistency of generated images. We train it before the GAN so the baseline run can log
these metrics.

In [ ]:
!python train_judge.py --out artifacts/judge.pt --epochs 25

## 2. Baseline conditional DCGAN
Default config: lr 2e-4 for both networks, β1 = 0.5, z = 64, ngf = ndf = 64, one-sided label smoothing 0.1, 1 D step per G step.
Logged every epoch: D and G loss, D accuracy, D(x), D(G(z)), G gradient norm, and a fixed-noise grid. Every 5 epochs we
also log the evaluation metrics (TensorBoard event files are written to `runs/baseline/tb`).

In [ ]:
!python train_gan.py --epochs {BASE_EPOCHS} --eval-every 5 --out runs/baseline

In [ ]:
show("runs/baseline/curves.png")
for ep in sorted({1, 10, BASE_EPOCHS // 2, BASE_EPOCHS}):
    print(f"epoch {ep}  (top row: normal, bottom row: pneumonia; SYNTHETIC)")
    show(f"runs/baseline/grids/epoch_{ep:03d}.png", width=420)

### 2b. A deliberately unstable configuration (diagnostics)
lr_D = 1e-3 (20× lr_G), 2 D steps per G step, no label smoothing. We expect the discriminator to overpower the generator:
D accuracy goes to 1, D(G(z)) goes to 0, and the generator stalls or collapses.

In [ ]:
!python train_gan.py --epochs {UNSTABLE_EPOCHS} --eval-every 5 --out runs/unstable --set lr_d=1e-3 lr_g=5e-5 label_smoothing=0.0 d_steps=2

In [ ]:
show("runs/unstable/curves.png")
show(f"runs/unstable/grids/epoch_{UNSTABLE_EPOCHS:03d}.png", width=420)

In [ ]:
# Optional: TensorBoard (renders in browser Colab; may not render inside Cursor). The PNG curves above carry the same data.
try:
    %load_ext tensorboard
    %tensorboard --logdir runs
except Exception as e:
    print("TensorBoard not available here:", e)

## 4. Ray Tune: distributed hyper-parameter search
Search space: lr_G, lr_D ~ loguniform(1e-5, 5e-4); β1 ∈ {0, 0.5}; z ∈ {32, 64, 128}; width ∈ {32, 64}; label smoothing ∈ {0, 0.1};
D steps ∈ {1, 2}; spectral norm ∈ {F, T}; batch ∈ {64, 128}.
ASHA early-stops weak trials on `quality_score`. Each trial reserves 0.25 GPU, so 4 trials share the A100.
Checkpoints are reported every iteration.

### 4.1 Smoke test

In [ ]:
!python tune_gan.py --num-samples 2 --max-epochs 2 --smoke

### 4.2 Full search

In [ ]:
print("samples:", TUNE_SAMPLES, "epochs:", TUNE_EPOCHS, "cpus per trial:", CPT)
!python tune_gan.py --num-samples {TUNE_SAMPLES} --max-epochs {TUNE_EPOCHS} --epochs-per-report 2 --gpus-per-trial 0.25 --cpus-per-trial {CPT}

In [ ]:
T = "outputs/tune_cgan_search"
print(json.dumps(json.load(open(f"{T}/tune_summary.json"))["best_config"], indent=1))
display(pd.read_csv(f"{T}/trials.csv").head(12))
show(f"{T}/trial_curves.png", f"{T}/hparam_scatter.png", f"{T}/resource_usage.png")
print("Best checkpoint grid (top: normal, bottom: pneumonia; SYNTHETIC)")
show(f"{T}/best_checkpoint_grid.png", width=420)

In [ ]:
# Ray resource log (stand-in for the dashboard screenshot, since port 8265 is not reachable from Cursor)
print(open(f"{T}/resource_log.csv").read()[:3000])
!ls ray_results/cgan_search | head -30

## 5. Select and stress-test the best model, and run the downstream comparison
We generate fixed-seed grids, latent interpolations, and same-z class swaps, then run the nearest-neighbour memorisation
check and look at failure cases. The downstream comparison uses 5 seeds on the held-out test set:
real-only | class-weighted | oversample+augment | real+synthetic (0.5×, 1× gap) | synthetic-only (TSTR).

In [ ]:
!python evaluate.py --checkpoint artifacts/best_generator.pt --out outputs/eval --seeds 5 --clf-epochs 20

In [ ]:
E = "outputs/eval"
show(f"{E}/grid_normal_seed1234.png", f"{E}/grid_pneumonia_seed1235.png", width=520)
show(f"{E}/real_vs_synthetic.png", f"{E}/class_swap_same_z.png", f"{E}/latent_interpolation.png")
show(f"{E}/nn_distance_hist.png", f"{E}/memorization_nearest_neighbors.png", f"{E}/failure_class_inconsistent.png")

In [ ]:
display(Markdown(open(f"{E}/downstream_tables.md").read()))
show(f"{E}/downstream_bars.png", f"{E}/confusion_matrices.png")
s = json.load(open(f"{E}/eval_summary.json"))
print(json.dumps({"gan_metrics": s["gan_metrics"], "memorization": s["memorization"]}, indent=1))

Comparison: the untuned baseline generator, evaluated with the same protocol (3 seeds).

In [ ]:
!python evaluate.py --checkpoint runs/baseline/generator.pt --out outputs/eval_baseline --seeds 3 --clf-epochs 20

### Reproducible fixed grid (same checkpoint + seed → same hash)

In [ ]:
import re, subprocess
def run_grid(out):
    o = subprocess.run([sys.executable, "regenerate_grid.py", "--checkpoint", "artifacts/best_generator.pt",
                        "--seed", "1234", "--out", out], capture_output=True, text=True).stdout
    print(o)
    return re.search(r"images sha256=(\w+)", o).group(1)
h1, h2 = run_grid("outputs/fixed_grid_run1.png"), run_grid("outputs/fixed_grid_run2.png")
json.dump({"hash_run1": h1, "hash_run2": h2, "identical": h1 == h2,
           "command": "python regenerate_grid.py --checkpoint artifacts/best_generator.pt --seed 1234"},
          open("outputs/repro.json", "w"), indent=1)
print("identical:", h1 == h2)
show("outputs/fixed_grid_run1.png", width=520)

## 6. Ray Serve deployment
The deployment uses `@serve.deployment` with autoscaling (1–4 replicas, target 4 ongoing requests per replica),
0.25 GPU and 1 CPU per replica, `@serve.batch`, a FastAPI ingress with pydantic validation, and responses marked as synthetic.

In [ ]:
import ray
from ray import serve
ray.shutdown()
ray.init(runtime_env={"env_vars": {"PYTHONPATH": PROJECT}}, ignore_reinit_error=True)
sys.path.insert(0, PROJECT)
from serve_app import deploy, replica_count
deploy("artifacts/best_generator.pt", num_gpus=0.25, max_replicas=4, num_cpus=SERVE_CPUS)
print("replicas:", replica_count())

In [ ]:
import requests
print(requests.get("http://127.0.0.1:8000/health").json())
print(json.dumps(requests.get("http://127.0.0.1:8000/model_card").json(), indent=1)[:1500])
!python client.py --url http://127.0.0.1:8000 --out outputs/serve
show("outputs/serve/api_grid_label0.png", "outputs/serve/api_grid_label1.png", width=420)

### Load test + autoscaling trace

In [ ]:
from load_test import run_load_test
df, trace = run_load_test("http://127.0.0.1:8000", levels=(1, 4, 16, 32, 64), duration_s=20, n_images=8,
                          probe=replica_count, out="outputs/serve")
display(df)
show("outputs/serve/load_test.png")

In [ ]:
# Text evidence from Ray (the dashboard at :8265 is not reachable from Cursor)
!serve status
!ray status
!ray list actors --limit 30

In [ ]:
serve.shutdown()
ray.shutdown()

## 7. Report, model card, and packaging

In [ ]:
!python make_report.py --out outputs/report
display(Markdown(open("outputs/report/REPORT.md").read().replace("](../../", "](")))

In [ ]:
import shutil
!cp report/MODEL_CARD.md outputs/report/
# Bundle results (small files only; full Ray trial dirs stay on the runtime)
!zip -qr hw3a_results.zip outputs artifacts/best_generator.pt artifacts/judge.pt artifacts/judge_metrics.json runs/baseline/history.json runs/baseline/curves.png runs/baseline/config.json runs/unstable/history.json runs/unstable/curves.png runs/baseline/tb runs/unstable/tb
!ls -lh hw3a_results.zip
try:
    from google.colab import files
    files.download("hw3a_results.zip")
except Exception as e:
    print("Automatic download unavailable here (", e, "). The executed notebook keeps all figures inline.")

In [ ]:
# Print the key numbers so they are saved in this notebook's outputs
summary = {
    "data": json.load(open("outputs/data/data_summary.json"))["splits"],
    "tune": json.load(open("outputs/tune_cgan_search/tune_summary.json")),
    "eval": {k: v for k, v in json.load(open("outputs/eval/eval_summary.json")).items() if k != "downstream"},
    "downstream_means": json.load(open("outputs/eval/eval_summary.json")).get("downstream", {}).get("means"),
    "load_test": pd.read_csv("outputs/serve/load_test.csv").to_dict(orient="records"),
    "repro": json.load(open("outputs/repro.json")),
}
print(json.dumps(summary, indent=1, default=str))